## https://github.com/House-Leo/FoundIR

## Feautres : Custom Tesseract loading : Removal Watershed Utility : Insert underscore for captured unknown akshara image at cursor position : Insert Akshara functionality at multiple positions/pages in a uploaded pdf : Add Restoration model (FoundIR) : Add AutoCompletion Feautre # Perform OCR on Image also.

#### https://stevehanov.ca/blog/index.php?id=114

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Python Library Functions
# ═══════════════════════════════════════════════════════════════════
import os
import io
import json
from PyQt5.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, 
                            QHBoxLayout, QLabel, QPushButton, QFileDialog, 
                            QScrollArea, QTextEdit, QSplitter, QSlider, 
                            QCheckBox, QMessageBox, QComboBox, QGroupBox,
                            QProgressBar, QShortcut, QLineEdit, QDialog) 
from PyQt5.QtGui import QPixmap, QImage, QPainter, QPen, QColor, QFont, QIcon, QKeySequence
from PyQt5.QtCore import Qt, QThread, pyqtSignal, QRect, QPoint, QTimer 
import fitz  # PyMuPDF
from PIL import Image
import numpy as np
import cv2
import sys
import speech_recognition as sr
from scipy.spatial import KDTree
import requests
import base64
from io import BytesIO
import time
from functools import lru_cache
from pathlib import Path

from word_suggestion_module import SuggestionManager
from word_suggestion_widget import add_suggestions_to_text_edit

# Add to imports at the top
from google.cloud import vision
from google.oauth2 import service_account
import io

from threading import Lock

# Custom Modules
from spell_check_module_V import (
    DictionaryManager,
    SpellChecker,
    SpellCheckHighlighter)

###################### Implement LRU cache or periodic cleanup
from functools import lru_cache
from collections import OrderedDict

from feedback_module_V import FeedbackDialog

class BoundedCache:
    def __init__(self, max_size=50):
        self.cache = OrderedDict()
        self.max_size = max_size
    
    def __setitem__(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.max_size:
            self.cache.popitem(last=False)

# Tesseract OCR imports
try:
    import pytesseract
    TESSERACT_AVAILABLE = True
except ImportError:
    TESSERACT_AVAILABLE = False
    print("Warning: pytesseract not available. OCR functionality will be disabled.")
    print("Install with: pip install pytesseract")

# Try to import imagehash for efficient similarity search
try:
    import imagehash
    IMAGEHASH_AVAILABLE = True
except ImportError:
    IMAGEHASH_AVAILABLE = False
    print("Warning: imagehash not available. Image comparison will be slower.")
    print("Install with: pip install imagehash")


# ═══════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════
CUSTOM_TESSDATA_PATH = None
FIXED_IMAGE_SIZE = (100, 100)
DEFAULT_ZOOM = 75
MIN_ZOOM = 50
MAX_ZOOM = 300
ZOOM_STEP = 25
BASE_RENDER_ZOOM = 2.0
MIN_LINE_HEIGHT = 10
MIN_LINE_WIDTH = 20
LINE_PADDING = 5
PAGE_CROP_SIZE = 1024
PAGE_CROP_STRIDE = 512
LINE_CROP_SIZE = 512
LINE_CROP_STRIDE = 256


# ═══════════════════════════════════════════════════════════════════
# ═══════════════════════════════════════════════════════════════════
# UTILITY FUNCTIONS
# ═══════════════════════════════════════════════════════════════════
# ═══════════════════════════════════════════════════════════════════

def ensure_directory(directory):
    """Ensure directory exists, create if not"""
    Path(directory).mkdir(parents=True, exist_ok=True)


def get_timestamp():
    """Get current timestamp in milliseconds"""
    return int(time.time() * 1000)


def qimage_to_numpy(qimage):
    """Convert QImage to numpy array efficiently"""
    width = qimage.width()
    height = qimage.height()
    
    # Convert to RGB32 format once
    buffer = qimage.convertToFormat(QImage.Format_RGB32)
    ptr = buffer.constBits()
    ptr.setsize(height * width * 4)
    
    # Create numpy array and extract RGB channels
    arr = np.frombuffer(ptr, np.uint8).reshape((height, width, 4))
    return arr[:, :, :3].copy()  # Return RGB only


def qimage_to_pil(qimage):
    """Convert QImage to PIL Image"""
    return Image.fromarray(qimage_to_numpy(qimage))


def save_image_safely(image, path):
    """Save image with atomic write operation"""
    temp_path = f"{path}.tmp"
    try:
        if isinstance(image, QPixmap):
            image = image.toImage()
        
        if not image.save(temp_path):
            raise IOError(f"Failed to save image to {temp_path}")
        
        if not os.path.exists(temp_path):
            raise IOError(f"Temporary file was not created: {temp_path}")
        
        # Atomic replace
        os.replace(temp_path, path)
        return path
    except Exception as e:
        if os.path.exists(temp_path):
            try:
                os.remove(temp_path)
            except:
                pass
        raise e

# ═══════════════════════════════════════════════════════════════════
# IMAGE LOADING UTILITY
# ═══════════════════════════════════════════════════════════════════
def load_image_as_pixmap(image_path):
    """
    Load an image file and convert it to QPixmap
    Supports: PNG, JPG, JPEG, BMP, TIFF, GIF
    """
    try:
        # Try loading directly with QPixmap first
        pixmap = QPixmap(image_path)
        
        if not pixmap.isNull():
            return pixmap
        
        # Fallback: Use PIL for more format support
        pil_image = Image.open(image_path)
        
        # Convert to RGB if necessary
        if pil_image.mode != 'RGB':
            pil_image = pil_image.convert('RGB')
        
        # Convert PIL to QImage
        img_array = np.array(pil_image)
        height, width, channel = img_array.shape
        bytes_per_line = 3 * width
        qimage = QImage(img_array.data, width, height, bytes_per_line, QImage.Format_RGB888)
        
        return QPixmap.fromImage(qimage.copy())
        
    except Exception as e:
        print(f"Error loading image: {e}")
        return None

# ═══════════════════════════════════════════════════════════════════
# ═══════════════════════════════════════════════════════════════════
# UTILITY CLASSES
# ═══════════════════════════════════════════════════════════════════
# ═══════════════════════════════════════════════════════════════════


# ═══════════════════════════════════════════════════════════════════
# FOUNDIR CLIENT
# ═══════════════════════════════════════════════════════════════════
class FoundIRClient:
    """Client for FoundIR denoising server with connection pooling"""
    
    def __init__(self, server_url='http://172.18.42.4:5000'):
        self.server_url = server_url
        self.enabled = False
        self.session = requests.Session()  # Connection pooling
        self.check_server_availability()
    
    def check_server_availability(self):
        """Check if FoundIR server is available"""
        try:
            response = self.session.get(f'{self.server_url}/health', timeout=2)
            if response.status_code == 200:
                self.enabled = True
                print("FoundIR server is available")
                return True
        except:
            pass
        self.enabled = False
        print("FoundIR server not available - denoising disabled")
        return False
    
    def _denoise_request(self, img_bytes, crop_size, crop_stride):
        """Common denoising request logic"""
        files = {'image': ('image.png', img_bytes, 'image/png')}
        data = {
            'crop_size': crop_size,
            'crop_stride': crop_stride
        }
        
        response = self.session.post(
            f'{self.server_url}/denoise_file',
            files=files,
            data=data,
            timeout=60
        )
        
        if response.status_code == 200:
            return Image.open(BytesIO(response.content))
        return None
    
    def denoise_qimage(self, qimage, crop_size=PAGE_CROP_SIZE, crop_stride=PAGE_CROP_STRIDE):
        """Denoise a QImage (for page-level denoising)"""
        if not self.enabled:
            return qimage
        
        try:
            # Convert to PIL and save to bytes
            pil_image = qimage_to_pil(qimage)
            img_bytes = BytesIO()
            pil_image.save(img_bytes, format='PNG')
            img_bytes.seek(0)
            
            # Send to server
            denoised_pil = self._denoise_request(img_bytes, crop_size, crop_stride)
            
            if denoised_pil:
                # Convert back to QImage
                denoised_array = np.array(denoised_pil)
                height, width, channel = denoised_array.shape
                bytes_per_line = 3 * width
                denoised_qimage = QImage(
                    denoised_array.data,
                    width,
                    height,
                    bytes_per_line,
                    QImage.Format_RGB888
                )
                return denoised_qimage.copy()
            else:
                print(f"Denoising failed")
                return qimage
                
        except Exception as e:
            print(f"Error in denoising QImage: {e}")
            return qimage
    
    def denoise_numpy(self, numpy_array, crop_size=LINE_CROP_SIZE, crop_stride=LINE_CROP_STRIDE):
        """Denoise a numpy array (for line-level denoising)"""
        if not self.enabled:
            return numpy_array
        
        try:
            # Convert to PIL and save to bytes
            pil_image = Image.fromarray(numpy_array.astype(np.uint8))
            img_bytes = BytesIO()
            pil_image.save(img_bytes, format='PNG')
            img_bytes.seek(0)
            
            # Send to server
            denoised_pil = self._denoise_request(img_bytes, crop_size, crop_stride)
            
            if denoised_pil:
                return np.array(denoised_pil)
            else:
                print(f"Denoising failed")
                return numpy_array
                
        except Exception as e:
            print(f"Error in denoising numpy: {e}")
            return numpy_array
    
    def close(self):
        """Close session"""
        self.session.close()


# ═══════════════════════════════════════════════════════════════════
# IMAGE PROCESSING UTILITIES
# ═══════════════════════════════════════════════════════════════════
class ImageProcessor:
    """Centralized image processing utilities"""
    
    @staticmethod
    def apply_adaptive_thresholding(gray_image):
        """Apply Otsu thresholding with adaptive adjustment"""
        if len(np.unique(gray_image)) <= 2:
            return gray_image
        
        # Apply Otsu's thresholding
        otsu_threshold, binary_img = cv2.threshold(
            gray_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        
        # Calculate mean intensity for adaptive adjustment
        mean_intensity = np.mean(gray_image)
        
        # Adjust threshold based on image brightness
        if mean_intensity > 200:
            threshold_value = min(otsu_threshold + 20, 200)
        elif mean_intensity > 150:
            threshold_value = min(otsu_threshold + 10, 180)
        else:
            threshold_value = max(otsu_threshold - 10, 100)
        
        # Apply adjusted threshold
        _, binary_img = cv2.threshold(
            gray_image, threshold_value, 255, cv2.THRESH_BINARY
        )
        
        print(f"Binarization: threshold={threshold_value} (Otsu={otsu_threshold}, Mean={mean_intensity:.1f})")
        return binary_img
    
    @staticmethod
    def ensure_black_text_on_white(binary_image):
        """Ensure text is black (0) on white (255) background"""
        white_pixels = np.sum(binary_image == 255)
        black_pixels = np.sum(binary_image == 0)
        
        if black_pixels > white_pixels:
            return cv2.bitwise_not(binary_image)
        return binary_image
    
    @staticmethod
    def segment_lines(binary_image):
        """Segment image into lines using horizontal projection"""
        # Create binary map: text = 1, background = 0
        binary_map = (binary_image == 0).astype(np.uint8)
        
        # Calculate horizontal projection
        horizontal_projection = np.sum(binary_map, axis=1)
        
        # Calculate adaptive threshold
        max_projection = np.max(horizontal_projection)
        non_zero_projection = horizontal_projection[horizontal_projection > 0]
        mean_projection = np.mean(non_zero_projection) if len(non_zero_projection) > 0 else 0
        
        threshold = max(
            max_projection * 0.2, 
            mean_projection * 0.5
        ) if max_projection > 0 else 0
        
        print(f"Line segmentation: max_proj={max_projection}, mean_proj={mean_projection:.1f}, threshold={threshold:.1f}")
        
        # Find line boundaries
        line_boundaries = []
        in_line = False
        start_row = 0
        
        for i, projection in enumerate(horizontal_projection):
            if projection > threshold and not in_line:
                start_row = i
                in_line = True
            elif projection <= threshold and in_line:
                line_boundaries.append((start_row, i))
                in_line = False
        
        if in_line:
            line_boundaries.append((start_row, len(horizontal_projection)))
        
        # Add padding
        padded_boundaries = []
        height = binary_image.shape[0]
        
        for start, end in line_boundaries:
            start_padded = max(0, start - LINE_PADDING)
            end_padded = min(height, end + LINE_PADDING)
            padded_boundaries.append((start_padded, end_padded))
        
        print(f"Detected {len(padded_boundaries)} lines")
        return padded_boundaries
    
    @staticmethod
    def trim_line_whitespace(line_image):
        """Trim horizontal whitespace from line image"""
        # Create binary map for text detection
        binary_map = (line_image == 0).astype(np.uint8)
        
        # Calculate vertical projection
        vertical_projection = np.sum(binary_map, axis=0)
        
        # Find non-empty columns
        non_empty_cols = np.where(vertical_projection > 0)[0]
        
        if len(non_empty_cols) == 0:
            return None
        
        # Trim to content
        left = non_empty_cols[0]
        right = non_empty_cols[-1] + 1
        trimmed = line_image[:, left:right]
        
        # Validate minimum size
        if trimmed.shape[0] < MIN_LINE_HEIGHT or trimmed.shape[1] < MIN_LINE_WIDTH:
            return None
        
        return trimmed


# ═══════════════════════════════════════════════════════════════════
# OCR WORKER
# ═══════════════════════════════════════════════════════════════════
class OCRWorker(QThread):
    """Worker thread to perform OCR without freezing the GUI"""
    result_ready = pyqtSignal(str, int, object, list) 
    google_vision_result_ready = pyqtSignal(str, int) 
    progress_update = pyqtSignal(int)
    error_occurred = pyqtSignal(str)
    
    def __init__(self, pixmap, page_num, language='hin+eng', config='', tessdata_path=None, 
                 spell_checker=None, parent=None, google_vision_client=None, 
                 predefined_line_segments=None): 
        super().__init__(parent)
        self.pixmap = pixmap
        self.page_num = page_num
        self.language = language
        self.config = config
        self.tessdata_path = tessdata_path
        self.spell_checker = spell_checker
        self.google_vision_client = google_vision_client
        self.predefined_line_segments = predefined_line_segments 
        self.image_processor = ImageProcessor()

    def save_image_to_disk(self, image_array, base_path, suffix):
        """Save image array to disk with error handling"""
        try:
            pil_image = Image.fromarray(image_array.astype(np.uint8))
            save_path = f"{base_path}_{suffix}.png"
            pil_image.save(save_path)
            print(f"Saved: {save_path}")
            return save_path
        except Exception as e:
            print(f"Error saving image {suffix}: {e}")
            return None
    
    def run(self):
        try:
            if not TESSERACT_AVAILABLE:
                self.error_occurred.emit("Tesseract is not available")
                return
            
            self.progress_update.emit(5)
            
            # Get parent reference
            parent = self.parent()
            foundir_enabled = (parent and hasattr(parent, 'foundir_enabled') and 
                            parent.foundir_enabled and parent.foundir_client.enabled)
            
            # Directory setup
            denoised_dir = parent.denoised_images_dir if parent else "Denoised_Images"
            pdf_name = os.path.splitext(parent.current_pdf_filename)[0] if parent else "document"
            timestamp = get_timestamp()
            page_base_path = os.path.join(denoised_dir, f"{pdf_name}_page{self.page_num + 1}")
            
            image = self.pixmap.toImage()
            
            # ═══════════════════════════════════════════════════════════════════
            # STEP 1: PAGE-LEVEL DENOISING
            # ═══════════════════════════════════════════════════════════════════
            if foundir_enabled:
                original_page_path = f"{page_base_path}_original_{timestamp}.png"
                image.save(original_page_path)
                print(f"Saved original page: {original_page_path}")
                
                self.progress_update.emit(10)
                if parent:
                    parent.statusBar().showMessage(f"Denoising page {self.page_num + 1} (full page)...")
                
                image = parent.foundir_client.denoise_qimage(image)
                
                denoised_page_path = f"{page_base_path}_denoised_{timestamp}.png"
                image.save(denoised_page_path)
                print(f"Saved denoised page: {denoised_page_path}")
                
                self.progress_update.emit(20)
            else:
                self.progress_update.emit(20)
            
            # Convert to numpy array
            arr = qimage_to_numpy(image)
            
            # Convert to grayscale
            gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            
            # Apply adaptive Otsu thresholding
            binary = self.image_processor.apply_adaptive_thresholding(gray)
            binary = self.image_processor.ensure_black_text_on_white(binary)
            
            self.progress_update.emit(25)
            
            # ═══════════════════════════════════════════════════════════════════
            # STEP 2: LINE SEGMENTATION - Use predefined or auto-detect
            # ═══════════════════════════════════════════════════════════════════
            if self.predefined_line_segments:
                # Use predefined line segments from saved JSON
                print(f"Using {len(self.predefined_line_segments)} predefined line segments")
                line_segments = []
                line_segment_rects = []
                
                for rect in self.predefined_line_segments:
                    # Convert QRect to (start_row, end_row) format
                    start_row = rect.y()
                    end_row = rect.y() + rect.height()
                    line_segments.append((start_row, end_row))
                    line_segment_rects.append(rect)
                
                if parent:
                    parent.statusBar().showMessage(
                        f"Using {len(line_segments)} saved line segments for page {self.page_num + 1}"
                    )
            else:
                # Auto-detect line segments
                print("Auto-detecting line segments")
                line_segments = self.image_processor.segment_lines(binary)
                
                # Store line segment rectangles for visualization
                line_segment_rects = []
                for start_row, end_row in line_segments:
                    rect = QRect(0, start_row, binary.shape[1], end_row - start_row)
                    line_segment_rects.append(rect)
                
                if parent:
                    parent.statusBar().showMessage(
                        f"Auto-detected {len(line_segments)} line segments for page {self.page_num + 1}"
                    )
            
            self.progress_update.emit(30)
            
            # Set custom tessdata path
            original_tessdata = None
            if self.tessdata_path and os.path.exists(self.tessdata_path):
                try:
                    original_tessdata = pytesseract.pytesseract.tesseract_cmd
                except:
                    pass
                
                os.environ['TESSDATA_PREFIX'] = self.tessdata_path
                print(f"Using custom tessdata path: {self.tessdata_path}")
                print(f"Using language model: {self.language}")
            
            try:
                # ═══════════════════════════════════════════════════════════════════
                # STEP 3: LINE-LEVEL PROCESSING
                # ═══════════════════════════════════════════════════════════════════
                all_text_lines = []
                total_lines = len(line_segments)
                
                for idx, (start_row, end_row) in enumerate(line_segments):
                    # Validate bounds
                    start_row = max(0, start_row)
                    end_row = min(binary.shape[0], end_row)
                    
                    if start_row >= end_row:
                        print(f"Skipping invalid line segment {idx + 1}: start={start_row}, end={end_row}")
                        continue
                    
                    # Extract line image
                    line_image_color = arr[start_row:end_row, :, :3]
                    line_image_binary = binary[start_row:end_row, :]
                    
                    # Trim whitespace
                    trimmed_binary = self.image_processor.trim_line_whitespace(line_image_binary)
                    
                    if trimmed_binary is None:
                        print(f"Skipping empty line segment {idx + 1}")
                        continue
                    
                    # Apply same trimming to color image
                    binary_map = (line_image_binary == 0).astype(np.uint8)
                    vertical_projection = np.sum(binary_map, axis=0)
                    non_empty_cols = np.where(vertical_projection > 0)[0]
                    
                    if len(non_empty_cols) > 0:
                        left = non_empty_cols[0]
                        right = non_empty_cols[-1] + 1
                        line_image_trimmed = line_image_color[:, left:right]
                    else:
                        line_image_trimmed = line_image_color
                    
                    # ═══════════════════════════════════════════════════════════════
                    # LINE-LEVEL DENOISING (if FoundIR is enabled)
                    # ═══════════════════════════════════════════════════════════════
                    if foundir_enabled:
                        line_base_path = f"{page_base_path}_line{idx + 1:03d}"
                        self.save_image_to_disk(
                            line_image_trimmed, 
                            line_base_path, 
                            f"original_{timestamp}"
                        )
                        
                        if parent:
                            parent.statusBar().showMessage(
                                f"Denoising page {self.page_num + 1}, line {idx + 1}/{total_lines}..."
                            )
                        
                        denoised_line = parent.foundir_client.denoise_numpy(
                            line_image_trimmed,
                            crop_size=LINE_CROP_SIZE,
                            crop_stride=LINE_CROP_STRIDE
                        )
                        
                        self.save_image_to_disk(
                            denoised_line, 
                            line_base_path, 
                            f"denoised_{timestamp}"
                        )
                        
                        pil_line = Image.fromarray(denoised_line.astype(np.uint8))
                    else:
                        # No denoising - use original line image
                        pil_line = Image.fromarray(line_image_trimmed)
                    
                    # ═══════════════════════════════════════════════════════════════
                    # PERFORM OCR ON LINE
                    # ═══════════════════════════════════════════════════════════════
                    line_config = '--psm 7'
                    line_text = pytesseract.image_to_string(
                        pil_line,
                        lang=self.language,
                        config=line_config
                    ).strip()
                    
                    if line_text:
                        all_text_lines.append(line_text)
                        print(f"Line {idx + 1}/{total_lines}: {line_text[:50]}...")
                    
                    # Update progress
                    progress = 30 + int((idx + 1) / total_lines * 60)
                    self.progress_update.emit(progress)
                
                # Combine all lines
                final_text = '\n'.join(all_text_lines)
                
                self.progress_update.emit(95)

                # ═══════════════════════════════════════════════════════════════
                # GOOGLE VISION OCR (if enabled)
                # ═══════════════════════════════════════════════════════════════
                parent = self.parent()
                if (parent and self.google_vision_client and 
                    self.google_vision_client.enabled and 
                    hasattr(parent, 'use_google_vision_comparison') and 
                    parent.use_google_vision_comparison):
                    
                    try:
                        google_text = self.google_vision_client.perform_ocr_from_pixmap(
                            self.pixmap,
                            language_hints=['hi', 'sa', 'en']
                        )
                        self.google_vision_result_ready.emit(google_text, self.page_num)
                    except Exception as e:
                        print(f"Google Vision OCR failed: {e}")
                
                # Spell check
                spell_check_results = None
                if self.spell_checker:
                    spell_check_results = self.spell_checker.check_text(final_text)

                self.result_ready.emit(final_text, self.page_num, spell_check_results, line_segment_rects)
                self.progress_update.emit(100)
                
            finally:
                # Restore original tessdata path
                if original_tessdata is not None:
                    pytesseract.pytesseract.tesseract_cmd = original_tessdata
                    if 'TESSDATA_PREFIX' in os.environ:
                        del os.environ['TESSDATA_PREFIX']
            
        except Exception as e:
            self.error_occurred.emit(f"OCR Error: {str(e)}")
            import traceback
            print(f"OCR Error Details:\n{traceback.format_exc()}")


# ═══════════════════════════════════════════════════════════════════
# MODEL DIALOG CLASSES
# ═══════════════════════════════════════════════════════════════════
class ModelConfigDialog(QDialog):
    """Dialog to configure custom Tesseract model path and settings"""
    
    def __init__(self, parent=None, current_tessdata_path=None, current_model_name=None):
        super().__init__(parent)
        self.tessdata_path = current_tessdata_path
        self.model_name = current_model_name or 'san'
        self.init_ui()
        
    def init_ui(self):
        """Initialize the dialog UI"""
        self.setWindowTitle("Tesseract Model Configuration")
        self.setMinimumWidth(600)
        self.setMinimumHeight(300)
        
        main_layout = QVBoxLayout()
        self.setLayout(main_layout)
        
        # Instructions
        instructions = QLabel(
            "Configure custom Tesseract-5 model for OCR.\n"
            "The tessdata folder should contain .traineddata files for your custom models."
        )
        instructions.setWordWrap(True)
        main_layout.addWidget(instructions)
        
        # Tessdata path section
        tessdata_group = QGroupBox("Tessdata Directory Path")
        tessdata_layout = QVBoxLayout()
        tessdata_group.setLayout(tessdata_layout)
        
        # Path selection layout
        path_layout = QHBoxLayout()
        
        self.path_input = QLineEdit()
        if self.tessdata_path:
            self.path_input.setText(self.tessdata_path)
        self.path_input.setPlaceholderText("Path to tessdata folder containing .traineddata files")
        path_layout.addWidget(self.path_input)
        
        self.browse_button = QPushButton("Browse...")
        self.browse_button.clicked.connect(self.browse_tessdata)
        path_layout.addWidget(self.browse_button)
        
        tessdata_layout.addLayout(path_layout)
        
        # Model verification
        self.verify_button = QPushButton("Verify Path")
        self.verify_button.clicked.connect(self.verify_tessdata_path)
        tessdata_layout.addWidget(self.verify_button)
        
        self.verification_label = QLabel("")
        self.verification_label.setWordWrap(True)
        tessdata_layout.addWidget(self.verification_label)
        
        main_layout.addWidget(tessdata_group)
        
        # Model selection section
        model_group = QGroupBox("Model Selection")
        model_layout = QVBoxLayout()
        model_group.setLayout(model_layout)
        
        model_layout.addWidget(QLabel("Select language/model:"))
        
        self.model_combo = QComboBox()
        self.model_combo.setEditable(True)
        self.model_combo.addItems([
            "san",
            "hin",
            "eng",
            "hin+eng",
            "san+eng",
        ])
        
        if self.model_name:
            index = self.model_combo.findText(self.model_name)
            if index >= 0:
                self.model_combo.setCurrentIndex(index)
            else:
                self.model_combo.setEditText(self.model_name)
        
        model_layout.addWidget(self.model_combo)
        
        # Custom model name input
        custom_layout = QHBoxLayout()
        custom_layout.addWidget(QLabel("Or enter custom model name:"))
        self.custom_model_input = QLineEdit()
        self.custom_model_input.setPlaceholderText("e.g., my_custom_model")
        custom_layout.addWidget(self.custom_model_input)
        model_layout.addLayout(custom_layout)
        
        main_layout.addWidget(model_group)
        
        # Additional OCR settings
        settings_group = QGroupBox("OCR Settings")
        settings_layout = QVBoxLayout()
        settings_group.setLayout(settings_layout)
        
        settings_layout.addWidget(QLabel("Page Segmentation Mode (PSM):"))
        self.psm_combo = QComboBox()
        self.psm_combo.addItems([
            "0 - Orientation and script detection (OSD) only",
            "1 - Automatic page segmentation with OSD",
            "3 - Fully automatic page segmentation, but no OSD (Default)",
            "4 - Assume a single column of text of variable sizes",
            "5 - Assume a single uniform block of vertically aligned text",
            "6 - Assume a single uniform block of text",
            "7 - Treat the image as a single text line",
            "8 - Treat the image as a single word",
            "9 - Treat the image as a single word in a circle",
            "10 - Treat the image as a single character",
            "11 - Sparse text. Find as much text as possible",
            "12 - Sparse text with OSD",
            "13 - Raw line. Treat the image as a single text line"
        ])
        self.psm_combo.setCurrentIndex(2)
        settings_layout.addWidget(self.psm_combo)
        
        main_layout.addWidget(settings_group)
        
        # Use default Tesseract option
        self.use_default_checkbox = QCheckBox("Use system default Tesseract (ignore custom path)")
        self.use_default_checkbox.toggled.connect(self.toggle_custom_path)
        main_layout.addWidget(self.use_default_checkbox)
        
        main_layout.addStretch()
        
        # Buttons
        buttons_layout = QHBoxLayout()
        
        self.cancel_button = QPushButton("Cancel")
        self.cancel_button.clicked.connect(self.reject)
        buttons_layout.addWidget(self.cancel_button)
        
        self.save_button = QPushButton("Save Configuration")
        self.save_button.clicked.connect(self.accept)
        buttons_layout.addWidget(self.save_button)
        
        main_layout.addLayout(buttons_layout)
    
    def browse_tessdata(self):
        """Open directory browser for tessdata path"""
        directory = QFileDialog.getExistingDirectory(
            self,
            "Select Tessdata Directory",
            self.path_input.text() or os.path.expanduser("~"),
            QFileDialog.ShowDirsOnly
        )
        
        if directory:
            self.path_input.setText(directory)
            self.verify_tessdata_path()
    
    def verify_tessdata_path(self):
        """Verify if the tessdata path is valid and contains model files"""
        path = self.path_input.text().strip()
        
        if not path:
            self.verification_label.setText("⚠ Please specify a tessdata path")
            self.verification_label.setStyleSheet("color: orange;")
            return False
        
        if not os.path.exists(path):
            self.verification_label.setText("❌ Path does not exist")
            self.verification_label.setStyleSheet("color: red;")
            return False
        
        if not os.path.isdir(path):
            self.verification_label.setText("❌ Path is not a directory")
            self.verification_label.setStyleSheet("color: red;")
            return False
        
        # Look for .traineddata files
        traineddata_files = [f for f in os.listdir(path) if f.endswith('.traineddata')]
        
        if not traineddata_files:
            self.verification_label.setText("⚠ No .traineddata files found in this directory")
            self.verification_label.setStyleSheet("color: orange;")
            return False
        
        # Success
        models_text = ", ".join([f.replace('.traineddata', '') for f in traineddata_files[:5]])
        if len(traineddata_files) > 5:
            models_text += f" and {len(traineddata_files) - 5} more"
        
        self.verification_label.setText(f"✓ Found {len(traineddata_files)} model(s): {models_text}")
        self.verification_label.setStyleSheet("color: green;")
        return True
    
    def toggle_custom_path(self, use_default):
        """Enable/disable custom path controls"""
        enabled = not use_default
        self.path_input.setEnabled(enabled)
        self.browse_button.setEnabled(enabled)
        self.verify_button.setEnabled(enabled)
        
        if use_default:
            self.verification_label.setText("Using system default Tesseract")
            self.verification_label.setStyleSheet("color: blue;")
    
    def get_config(self):
        """Get the configuration from the dialog"""
        if self.use_default_checkbox.isChecked():
            tessdata_path = None
        else:
            tessdata_path = self.path_input.text().strip() or None
        
        # Get model name
        custom_model = self.custom_model_input.text().strip()
        if custom_model:
            model_name = custom_model
        else:
            model_name = self.model_combo.currentText()
        
        # Get PSM value
        psm_text = self.psm_combo.currentText()
        psm_value = int(psm_text.split('-')[0].strip())
        
        return {
            'tessdata_path': tessdata_path,
            'model_name': model_name,
            'psm': psm_value
        }

# ═══════════════════════════════════════════════════════════════════
# IMAGE COMPARISON DIALOG CLASSES
# ═══════════════════════════════════════════════════════════════════
class ImageComparisonDialog(QDialog):
    """Dialog to display similar images and choose insertion options"""
    
    def __init__(self, parent=None, selected_images=None, similar_images=None):
        super().__init__(parent)
        self.selected_images = selected_images or []
        self.similar_images = similar_images or []
        self.selected_similar_images = []
        self.is_insert_underscore = True
        self.init_ui()
        
    def init_ui(self):
        """Initialize the dialog UI"""
        self.setWindowTitle("Similar Images")
        self.setMinimumWidth(700)
        self.setMinimumHeight(500)
        
        main_layout = QVBoxLayout()
        self.setLayout(main_layout)
        
        # Instructions
        instructions = QLabel("Your selections are highlighted with green borders. Select similar images to insert.")
        main_layout.addWidget(instructions)
        
        # Create scroll area
        scroll_area = QScrollArea()
        scroll_area.setWidgetResizable(True)
        scroll_content = QWidget()
        scroll_layout = QVBoxLayout(scroll_content)
        
        # Selected images section
        selected_group = QGroupBox("Your Selected Aksharas")
        selected_layout = QHBoxLayout()
        selected_group.setLayout(selected_layout)
        
        # Add all selected images with green border
        for selected_pixmap in self.selected_images:
            selected_container = QWidget()
            selected_container_layout = QVBoxLayout(selected_container)
            
            selected_label = QLabel()
            selected_label.setPixmap(selected_pixmap)
            selected_label.setStyleSheet("border: 3px solid green;")
            selected_label.setFixedSize(selected_pixmap.width() + 6, selected_pixmap.height() + 6)
            selected_container_layout.addWidget(selected_label)
            
            selected_layout.addWidget(selected_container)
        
        scroll_layout.addWidget(selected_group)
        
        # Similar images section
        similar_group = QGroupBox("Similar Aksharas Found")
        similar_layout = QHBoxLayout()
        similar_group.setLayout(similar_layout)
        
        parent_widget = self.parent()
        
        # Add similar images with checkable frames
        self.image_buttons = []
        
        for i, (similarity, image, image_id) in enumerate(self.similar_images):
            # Create container
            image_container = QWidget()
            container_layout = QVBoxLayout(image_container)
            
            # Create checkable button
            image_button = QPushButton()
            image_button.setCheckable(True)
            image_button.setFixedSize(image.width() + 10, image.height() + 10)
            image_button.setIcon(QIcon(image))
            image_button.setIconSize(image.size())
            image_button.clicked.connect(lambda checked, idx=i: self.toggle_image_selection(idx))
            container_layout.addWidget(image_button)
            
            # Add similarity score
            sim_text = QLabel(f"{similarity:.2f}")
            sim_text.setAlignment(Qt.AlignCenter)
            container_layout.addWidget(sim_text)
            
            # Add page number
            if parent_widget and hasattr(parent_widget, 'image_lookup_data') and image_id in parent_widget.image_lookup_data:
                page_num = parent_widget.image_lookup_data[image_id].get('page', '?')
                page_label = QLabel(f"Page {page_num}")
                page_label.setAlignment(Qt.AlignCenter)
                page_label.setStyleSheet("font-size: 9px; color: #666;")
                container_layout.addWidget(page_label)
            
            similar_layout.addWidget(image_container)
            self.image_buttons.append((image_button, image_id))
        
        scroll_layout.addWidget(similar_group)
        
        scroll_area.setWidget(scroll_content)
        main_layout.addWidget(scroll_area)
        
        # Insertion options
        options_layout = QHBoxLayout()
        
        self.underscore_radio = QCheckBox("Insert underscore (_)")
        self.underscore_radio.setChecked(True)
        self.underscore_radio.toggled.connect(self.update_char_input_state)
        options_layout.addWidget(self.underscore_radio)
        
        self.character_radio = QCheckBox("Insert akshara")
        self.character_radio.toggled.connect(self.update_char_input_state)
        options_layout.addWidget(self.character_radio)
        
        options_layout.addWidget(QLabel("Akshara:"))
        self.char_input = QLineEdit()
        self.char_input.setMaxLength(5)
        self.char_input.setFixedWidth(50)
        self.char_input.setEnabled(False)
        options_layout.addWidget(self.char_input)
        
        main_layout.addLayout(options_layout)
        
        # Buttons
        buttons_layout = QHBoxLayout()
        
        self.cancel_button = QPushButton("Cancel")
        self.cancel_button.clicked.connect(self.reject)
        buttons_layout.addWidget(self.cancel_button)
        
        self.insert_button = QPushButton("Insert Selected")
        self.insert_button.clicked.connect(self.accept)
        buttons_layout.addWidget(self.insert_button)
        
        main_layout.addLayout(buttons_layout)

    def update_char_input_state(self):
        """Enable or disable character input"""
        self.char_input.setEnabled(self.character_radio.isChecked())
        if self.character_radio.isChecked():
            self.char_input.setFocus()
    
    def toggle_image_selection(self, index):
        """Toggle image selection and update button styles"""
        button, image_id = self.image_buttons[index]
        if button.isChecked():
            button.setStyleSheet("background-color: lightblue;")
            if image_id not in self.selected_similar_images:
                self.selected_similar_images.append(image_id)
        else:
            button.setStyleSheet("")
            if image_id in self.selected_similar_images:
                self.selected_similar_images.remove(image_id)
    
    def get_results(self):
        """Get the dialog results"""
        is_insert_underscore = self.underscore_radio.isChecked()
        char_to_insert = "_"
        
        if not is_insert_underscore:
            input_char = self.char_input.text()
            if input_char:
                char_to_insert = input_char[0]
        
        return {
            'selected_images': self.selected_similar_images,
            'insert_underscore': is_insert_underscore,
            'character': char_to_insert
        }


# ═══════════════════════════════════════════════════════════════════
# SPEECH RECOGNITION WORKER
# ═══════════════════════════════════════════════════════════════════
class SpeechRecognitionWorker(QThread):
    """Worker thread for speech recognition"""
    result_ready = pyqtSignal(str)
    error_occurred = pyqtSignal(str)
    status_update = pyqtSignal(str)
    
    def __init__(self, language='en-US', timeout=5, phrase_time_limit=None):
        super().__init__()
        self.language = language
        self.timeout = timeout
        self.phrase_time_limit = phrase_time_limit
        self.listening = False
        self.recognizer = sr.Recognizer()
        self._stop_requested = False
        
        # Adjust recognition parameters
        if 'hi' in language.lower():
            self.recognizer.energy_threshold = 300
            self.recognizer.pause_threshold = 1.0
        
        self.recognizer.dynamic_energy_threshold = True
        
    def run(self):
        """Start listening and recognition"""
        try:
            self.listening = True
            self._stop_requested = False
            self.status_update.emit("Listening...")
            
            with sr.Microphone() as source:
                self.recognizer.adjust_for_ambient_noise(source, duration=1)
                
                if self._stop_requested:
                    return
                    
                self.status_update.emit("Speak now...")
                
                try:
                    audio = self.recognizer.listen(
                        source, 
                        timeout=self.timeout,
                        phrase_time_limit=self.phrase_time_limit
                    )
                    
                    if self._stop_requested:
                        return
                    
                    self.status_update.emit("Processing speech...")
                    
                    text = self.recognizer.recognize_google(audio, language=self.language)
                    print(f"Raw recognized text: {text}")
                    self.result_ready.emit(text)
                    
                except sr.WaitTimeoutError:
                    self.error_occurred.emit("No speech detected within timeout period")
                except sr.UnknownValueError:
                    self.error_occurred.emit("Could not understand audio")
                except sr.RequestError as e:
                    self.error_occurred.emit(f"Could not request results; {str(e)}")
        except Exception as e:
            self.error_occurred.emit(f"Error in speech recognition: {str(e)}")
        finally:
            self.listening = False
    
    def stop(self):
        """Request the worker to stop"""
        self._stop_requested = True


# ═══════════════════════════════════════════════════════════════════
# PDF VIEW LABEL
# ═══════════════════════════════════════════════════════════════════
class PDFViewLabel(QLabel):
    """Custom label for PDF viewing and selection"""
    selectionChanged = pyqtSignal(QRect)
    selectionFinished = pyqtSignal(QRect)
    lineSegmentChanged = pyqtSignal(int, QRect)
    newLineSegmentCreated = pyqtSignal(QRect)  # New signal
    
    def __init__(self, parent=None):
        super().__init__(parent)
        self.selection_mode = False
        self.drawing = False
        self.start_point = QPoint()
        self.current_rect = QRect()
        self.user_selections = []
        self.user_selections_original = []
        self.line_segments = []
        self.line_segment_editing_mode = False
        self.add_line_segment_mode = False  # New mode
        self.selected_line_index = -1
        self.resize_mode = None
        self.resize_start_point = QPoint()
        self.resize_original_rect = QRect()
        self.setCursor(Qt.ArrowCursor)
    
    def enable_selection(self, enabled):
        self.selection_mode = enabled
        self.setCursor(Qt.CrossCursor if enabled else Qt.ArrowCursor)
        if not enabled:
            self.current_rect = QRect()
            self.update()
    
    def enable_line_segment_editing(self, enabled):
        """Enable or disable line segment editing mode"""
        self.line_segment_editing_mode = enabled
        if enabled:
            self.selection_mode = False
            self.add_line_segment_mode = False  # Disable add mode
            self.setCursor(Qt.ArrowCursor)
        self.selected_line_index = -1
        self.update()
    
    def enable_add_line_segment(self, enabled):
        """Enable or disable add line segment mode"""
        self.add_line_segment_mode = enabled
        if enabled:
            self.selection_mode = False
            self.line_segment_editing_mode = False
            self.selected_line_index = -1
            self.setCursor(Qt.CrossCursor)
        else:
            self.setCursor(Qt.ArrowCursor)
            self.current_rect = QRect()
        self.update()

    def add_user_selection(self, rect, original_rect=None):
        """Add a user selection to the list"""
        if not rect.isEmpty():
            self.user_selections.append(rect)
            self.user_selections_original.append(original_rect if original_rect is not None else rect)
            self.update()
            return True
        return False
    
    def clear_user_selections(self):
        """Clear all user selections"""
        self.user_selections = []
        self.user_selections_original = []
        self.update()
    
    def set_line_segments(self, line_segments):
        """Set line segments to display"""
        self.line_segments = line_segments
        self.update()

    def clear_line_segments(self):
        """Clear line segments"""
        self.line_segments = []
        self.selected_line_index = -1
        self.update()
    
    def add_new_line_segment(self, rect):
        """Add a new line segment"""
        if not rect.isEmpty():
            self.line_segments.append(rect)
            self.update()
            return True
        return False
    
    def delete_line_segment(self, index):
        """Delete a line segment by index"""
        if 0 <= index < len(self.line_segments):
            del self.line_segments[index]
            self.selected_line_index = -1
            self.update()
            return True
        return False
    
    def get_line_segment_at_point(self, point):
        """Get line segment index at given point"""
        for i, rect in enumerate(self.line_segments):
            if rect.contains(point):
                return i
        return -1
    
    def get_resize_mode(self, point, rect):
        """Determine resize mode based on cursor position"""
        margin = 8  # Pixels from edge to trigger resize
        
        if abs(point.y() - rect.top()) < margin:
            return 'top'
        elif abs(point.y() - rect.bottom()) < margin:
            return 'bottom'
        elif abs(point.x() - rect.left()) < margin:
            return 'left'
        elif abs(point.x() - rect.right()) < margin:
            return 'right'
        elif rect.contains(point):
            return 'move'
        return None
    
    def update_cursor_for_resize_mode(self, mode):
        """Update cursor based on resize mode"""
        if mode == 'top' or mode == 'bottom':
            self.setCursor(Qt.SizeVerCursor)
        elif mode == 'left' or mode == 'right':
            self.setCursor(Qt.SizeHorCursor)
        elif mode == 'move':
            self.setCursor(Qt.SizeAllCursor)
        else:
            self.setCursor(Qt.ArrowCursor)
    
    def mousePressEvent(self, event):
        if self.add_line_segment_mode and event.button() == Qt.LeftButton:
            # Add new line segment mode - start drawing
            self.drawing = True
            self.start_point = event.pos()
            self.current_rect = QRect()
            self.update()
        
        elif self.line_segment_editing_mode and event.button() == Qt.LeftButton:
            # Line segment editing mode
            line_index = self.get_line_segment_at_point(event.pos())
            
            if line_index >= 0:
                self.selected_line_index = line_index
                rect = self.line_segments[line_index]
                self.resize_mode = self.get_resize_mode(event.pos(), rect)
                self.resize_start_point = event.pos()
                self.resize_original_rect = QRect(rect)
                self.update()
            else:
                self.selected_line_index = -1
                self.resize_mode = None
                self.update()
        
        elif self.selection_mode and event.button() == Qt.LeftButton:
            # Regular selection mode
            self.drawing = True
            self.start_point = event.pos()
            self.current_rect = QRect()
            self.update()
    
    def mouseMoveEvent(self, event):
        if self.add_line_segment_mode and self.drawing:
            # Drawing new line segment
            self.current_rect = QRect(self.start_point, event.pos()).normalized()
            self.update()
        
        elif self.line_segment_editing_mode:
            if self.selected_line_index >= 0 and self.resize_mode:
                # Resize or move the selected line segment
                rect = self.line_segments[self.selected_line_index]
                delta = event.pos() - self.resize_start_point
                
                new_rect = QRect(self.resize_original_rect)
                
                if self.resize_mode == 'top':
                    new_rect.setTop(self.resize_original_rect.top() + delta.y())
                elif self.resize_mode == 'bottom':
                    new_rect.setBottom(self.resize_original_rect.bottom() + delta.y())
                elif self.resize_mode == 'left':
                    new_rect.setLeft(self.resize_original_rect.left() + delta.x())
                elif self.resize_mode == 'right':
                    new_rect.setRight(self.resize_original_rect.right() + delta.x())
                elif self.resize_mode == 'move':
                    new_rect.translate(delta)
                
                # Ensure minimum size
                if new_rect.width() >= 20 and new_rect.height() >= 10:
                    self.line_segments[self.selected_line_index] = new_rect
                    self.update()
            else:
                # Update cursor based on hover
                line_index = self.get_line_segment_at_point(event.pos())
                if line_index >= 0:
                    rect = self.line_segments[line_index]
                    mode = self.get_resize_mode(event.pos(), rect)
                    self.update_cursor_for_resize_mode(mode)
                else:
                    self.setCursor(Qt.ArrowCursor)
        
        elif self.selection_mode and self.drawing:
            self.current_rect = QRect(self.start_point, event.pos()).normalized()
            self.selectionChanged.emit(self.current_rect)
            self.update()
    
    def mouseReleaseEvent(self, event):
        if self.add_line_segment_mode and self.drawing and event.button() == Qt.LeftButton:
            # Finalize new line segment
            self.drawing = False
            self.current_rect = QRect(self.start_point, event.pos()).normalized()
            
            if not self.current_rect.isEmpty() and self.current_rect.width() >= 20 and self.current_rect.height() >= 10:
                self.newLineSegmentCreated.emit(self.current_rect)
                self.current_rect = QRect()
            else:
                self.current_rect = QRect()
            
            self.update()
        
        elif self.line_segment_editing_mode and event.button() == Qt.LeftButton:
            if self.selected_line_index >= 0 and self.resize_mode:
                # Emit signal with updated line segment
                rect = self.line_segments[self.selected_line_index]
                self.lineSegmentChanged.emit(self.selected_line_index, rect)
                self.resize_mode = None
        
        elif self.selection_mode and self.drawing and event.button() == Qt.LeftButton:
            self.drawing = False
            self.current_rect = QRect(self.start_point, event.pos()).normalized()
            self.selectionFinished.emit(self.current_rect)
            self.update()
    
    def paintEvent(self, event):
        super().paintEvent(event)
        
        painter = QPainter(self)
        
        # Draw line segments (blue boxes)
        if self.line_segments:
            for i, box in enumerate(self.line_segments):
                if i == self.selected_line_index and self.line_segment_editing_mode:
                    # Highlight selected line segment
                    pen = QPen(QColor(255, 165, 0, 255), 3)  # Orange, thicker
                    painter.setPen(pen)
                    painter.drawRect(box)
                    painter.fillRect(box, QColor(255, 165, 0, 30))
                else:
                    pen = QPen(QColor(0, 120, 215, 200), 2)  # Blue
                    painter.setPen(pen)
                    painter.drawRect(box)
                
                # Draw line number
                painter.drawText(box.topLeft() + QPoint(5, 15), f"L{i+1}")
        
        # Draw new line segment being created (green dashed)
        if self.add_line_segment_mode and not self.current_rect.isEmpty():
            pen = QPen(QColor(0, 255, 0, 200), 2, Qt.DashLine)
            painter.setPen(pen)
            painter.drawRect(self.current_rect)
            painter.fillRect(self.current_rect, QColor(0, 255, 0, 40))
        
        # Draw user selections (green boxes)
        if self.user_selections:
            pen = QPen(QColor(0, 180, 0, 180), 2)
            painter.setPen(pen)
            
            for box in self.user_selections:
                painter.drawRect(box)
                painter.fillRect(box, QColor(0, 180, 0, 30))
        
        # Draw manual selection (red box)
        if self.selection_mode and not self.current_rect.isEmpty():
            pen = QPen(QColor(255, 0, 0, 180), 2)
            painter.setPen(pen)
            painter.drawRect(self.current_rect)
            painter.fillRect(self.current_rect, QColor(255, 0, 0, 60))

# ============================================================================
# GOOGLE VISION API OCR
# ============================================================================

class GoogleVisionOCR:
    """Google Vision API OCR client for text comparison"""
    
    def __init__(self, credentials_path: str = None):
        self.client = None
        self.enabled = False
        
        try:
            from google.cloud import vision
            from google.oauth2 import service_account
            
            if credentials_path and os.path.exists(credentials_path):
                credentials = service_account.Credentials.from_service_account_file(
                    credentials_path
                )
                self.client = vision.ImageAnnotatorClient(credentials=credentials)
            else:
                self.client = vision.ImageAnnotatorClient()
            
            self.enabled = True
            print("✓ Google Vision API initialized")
            
        except Exception as e:
            self.enabled = False
            print(f"⚠ Google Vision API not available: {e}")
    
    def perform_ocr_from_pixmap(self, pixmap, language_hints=None):
        """Perform OCR on QPixmap"""
        if not self.enabled:
            return ""
        
        try:
            # Convert QPixmap to bytes
            buffer = io.BytesIO()
            qimage = pixmap.toImage()
            qimage.save(buffer, "PNG")
            image_bytes = buffer.getvalue()
            
            # Import here to avoid dependency issues
            from google.cloud import vision
            
            # Create Vision API image
            image = vision.Image(content=image_bytes)
            
            # Configure language hints
            image_context = None
            if language_hints:
                image_context = vision.ImageContext(language_hints=language_hints)
            
            # Perform OCR
            if image_context:
                response = self.client.text_detection(image=image, image_context=image_context)
            else:
                response = self.client.text_detection(image=image)
            
            # Extract text
            if response.text_annotations:
                return response.text_annotations[0].description
            return ""
            
        except Exception as e:
            print(f"Google Vision OCR error: {e}")
            return ""

# ============================================================================
# GOOGLE VISION API OCR WORKER
# ============================================================================

class GoogleVisionOCRWorker(QThread):
    """Worker thread for Google Vision API OCR"""
    result_ready = pyqtSignal(str, int)  # text, page_num
    error_occurred = pyqtSignal(str)
    
    def __init__(self, pixmap, page_num, google_vision_client, language_hints=None):
        super().__init__()
        self.pixmap = pixmap
        self.page_num = page_num
        self.google_vision_client = google_vision_client
        self.language_hints = language_hints or ['hi', 'sa', 'en']
    
    def run(self):
        try:
            text = self.google_vision_client.perform_ocr_from_pixmap(
                self.pixmap, 
                self.language_hints
            )
            self.result_ready.emit(text, self.page_num)
        except Exception as e:
            self.error_occurred.emit(f"Google Vision OCR Error: {str(e)}")

# ═══════════════════════════════════════════════════════════════════
# MAIN APPLICATION
# ═══════════════════════════════════════════════════════════════════
class PDFOCRViewer(QMainWindow):
    # ═══════════════════════════════════════════════════════════════════
    # MAIN AAPLICATION INITIALIZER
    # ═══════════════════════════════════════════════════════════════════
    def __init__(self):
        super().__init__()

        # Initialize FoundIR client
        self.foundir_client = FoundIRClient('http://172.18.42.4:5000')
        self.foundir_enabled = False

        # Initialize Google Vision OCR client (NEW)
        self.google_vision_client = None
        self.use_google_vision_comparison = False
        self.google_vision_cache = {}
        self.init_google_vision_ocr()
        
        # Directories
        self.denoised_images_dir = "Denoised_Images"
        self.image_lookup_dir = "Image_Lookup"
        ensure_directory(self.denoised_images_dir)
        ensure_directory(self.image_lookup_dir)

        # ═══════════════════════════════════════════════════════════════
        # Document type tracking
        # ═══════════════════════════════════════════════════════════════
        self.current_document_type = None  # 'pdf' or 'image'
        self.current_image_path = None
        
        # Initialize data structures
        self.pdf_document = None
        self.current_page = 0
        self.total_pages = 0
        self.ocr_workers = []
        self.zoom_factor = DEFAULT_ZOOM / 100.0
        self.curr_page_pixmap = None
        self.current_selection = None
        self.current_selection_original = None
        self.current_pdf_filename = ""
        
        # OCR settings
        self.auto_ocr_enabled = False
        self.ocr_language = 'san'
        self.ocr_config = '--psm 6'
        self.tessdata_path = None
        self.ocr_psm = 6
        
        # OCR cache
        self.ocr_cache = {}
        self.line_segments_cache = {} 
        self.line_segments_dir = "Line_Segments"
        ensure_directory(self.line_segments_dir)
        self.line_segment_editing_mode = False
        self.selected_line_segment_index = -1
        self.add_line_segment_mode = False
        self.ocr_in_progress = set()
        self.page_selections = {}
        
        # Image lookup
        self.image_lookup_file = os.path.join(self.image_lookup_dir, "image_lookup.json")
        self.next_image_id = 1
        self.image_lookup_data = {}
        self.hash_tree = None
        self.hash_ids = []
        self.load_image_lookup_data()
        
        if IMAGEHASH_AVAILABLE and self.image_lookup_data:
            self.build_hash_index()
        
        # Speech recognition
        self.speech_worker = None
        self.speech_language = "hi-IN"
        self.speech_timeout = 10
        self.speech_phrase_limit = 10

        # Spell check
        self.dict_manager = DictionaryManager("Dictionaries")
        self.spell_checker = SpellChecker(self.dict_manager)
        self.highlighter = None
    
        # ═══════════════════════════════════════════════════════════════
        # Initialize word suggestion manager
        # ═══════════════════════════════════════════════════════════════
        self.suggestion_manager = SuggestionManager("Dictionaries")
        self.suggestions_enabled = True
        
        # Initialize UI
        self.init_ui()
        self.load_ocr_config()
        self.init_text_editor_context_menu()
        
    # ═══════════════════════════════════════════════════════════════════
    # MAIN APPLICATION UI MANAGER 
    # ═══════════════════════════════════════════════════════════════════
    def init_ui(self):
        self.setWindowTitle("Pāṇḍulipisaṃśodhaka")
        self.setGeometry(100, 100, 1400, 900)

        # ═══════════════════════════════════════════════════════════════
        # MENU BAR
        # ═══════════════════════════════════════════════════════════════

        # Create menu bar
        menubar = self.menuBar()
        
        # Help menu
        help_menu = menubar.addMenu("Help")
        
        feedback_action = help_menu.addAction("📧 Send Feedback")
        feedback_action.triggered.connect(self.show_feedback_dialog)
        feedback_action.setShortcut("Ctrl+Shift+F")
        
        help_menu.addSeparator()
        
        about_action = help_menu.addAction("About PANDULIPI")
        
        # Main widget
        main_widget = QWidget()
        main_layout = QVBoxLayout()
        self.main_layout = main_layout
        main_widget.setLayout(main_layout)
        self.setCentralWidget(main_widget)
        
        # ═══════════════════════════════════════════════════════════════
        # PDF/Image Controls
        # ═══════════════════════════════════════════════════════════════
        pdf_group = QGroupBox("Document Controls")
        pdf_layout = QHBoxLayout()
        pdf_layout.setContentsMargins(10, 5, 10, 5)
        pdf_group.setLayout(pdf_layout)

        # Changed button text to be more generic
        self.load_button = QPushButton("Load PDF/Image")
        self.load_button.clicked.connect(self.load_document)
        self.load_button.setToolTip("Load PDF or Image file (PNG, JPG, JPEG, BMP, TIFF)")
        pdf_layout.addWidget(self.load_button)

        self.prev_button = QPushButton("Previous Page")
        self.prev_button.clicked.connect(self.prev_page)
        self.prev_button.setEnabled(False)
        pdf_layout.addWidget(self.prev_button)

        self.page_label = QLabel("Page 0 of 0")
        pdf_layout.addWidget(self.page_label)

        self.next_button = QPushButton("Next Page")
        self.next_button.clicked.connect(self.next_page)
        self.next_button.setEnabled(False)
        pdf_layout.addWidget(self.next_button)

        # ═══════════════════════════════════════════════════════════════════
        # SPELL CHECK CONTROLS
        # ═══════════════════════════════════════════════════════════════════
        self.spell_check_enabled_checkbox = QCheckBox("Enable Spell Check")
        self.spell_check_enabled_checkbox.setChecked(False)  # Disabled by default
        self.spell_check_enabled_checkbox.toggled.connect(self.toggle_spell_check)
        pdf_layout.addWidget(self.spell_check_enabled_checkbox)

        self.show_legend_button = QPushButton("Color Legend")
        self.show_legend_button.clicked.connect(self.show_spell_check_legend)
        self.show_legend_button.setEnabled(False)  # ✅ Disabled initially
        pdf_layout.addWidget(self.show_legend_button)

        self.dict_stats_button = QPushButton("Dictionary Stats")
        self.dict_stats_button.clicked.connect(self.show_dictionary_stats)
        self.dict_stats_button.setEnabled(False)  # ✅ Disabled initially
        pdf_layout.addWidget(self.dict_stats_button)

        self.add_to_dict_button = QPushButton("Add to Dictionary")
        self.add_to_dict_button.clicked.connect(self.add_selected_to_dictionary)
        self.add_to_dict_button.setEnabled(False)  # ✅ Disabled initially
        pdf_layout.addWidget(self.add_to_dict_button)

        pdf_layout.addStretch()
        main_layout.addWidget(pdf_group)
        
        # OCR Controls
        ocr_group = QGroupBox("OCR Settings")
        ocr_layout = QHBoxLayout()
        ocr_layout.setContentsMargins(10, 5, 10, 5)
        ocr_group.setLayout(ocr_layout)
        
        ocr_layout.addWidget(QLabel("Language:"))
        self.ocr_language_combo = QComboBox()
        self.ocr_language_combo.addItems([
            "Sanskrit (san)",
            "Hindi + English (hin+eng)",
            "Hindi (hin)",
            "English (eng)"
        ])
        self.ocr_language_combo.currentIndexChanged.connect(self.change_ocr_language)
        self.ocr_language_combo.setEnabled(TESSERACT_AVAILABLE)
        ocr_layout.addWidget(self.ocr_language_combo)
        
        self.model_config_button = QPushButton("Configure Model")
        self.model_config_button.clicked.connect(self.open_model_config)
        self.model_config_button.setEnabled(TESSERACT_AVAILABLE)
        self.model_config_button.setToolTip("Configure custom Tesseract-5 model")
        ocr_layout.addWidget(self.model_config_button)
        
        self.manual_ocr_button = QPushButton("Run OCR Now")
        self.manual_ocr_button.clicked.connect(self.save_line_segments_to_file)
        self.manual_ocr_button.clicked.connect(self.run_manual_ocr)
        self.manual_ocr_button.setEnabled(False)
        ocr_layout.addWidget(self.manual_ocr_button)
        
        self.ocr_progress = QProgressBar()
        self.ocr_progress.setRange(0, 100)
        self.ocr_progress.setValue(0)
        self.ocr_progress.setVisible(False)
        ocr_layout.addWidget(self.ocr_progress, 1)

        self.foundir_checkbox = QCheckBox("Use FoundIR Denoising")
        self.foundir_checkbox.setChecked(True)
        self.foundir_checkbox.toggled.connect(self.toggle_foundir)
        self.foundir_checkbox.setEnabled(self.foundir_client.enabled)
        if not self.foundir_client.enabled:
            self.foundir_checkbox.setToolTip("FoundIR server not available")
        ocr_layout.addWidget(self.foundir_checkbox)

        self.foundir_status_label = QLabel("⚠ Server offline")
        self.foundir_status_label.setStyleSheet("color: orange;")
        if self.foundir_client.enabled:
            self.foundir_status_label.setText("✓ Server ready")
            self.foundir_status_label.setStyleSheet("color: green;")
        ocr_layout.addWidget(self.foundir_status_label)

        self.foundir_reconnect_button = QPushButton("Reconnect")
        self.foundir_reconnect_button.clicked.connect(self.reconnect_foundir)
        self.foundir_reconnect_button.setEnabled(not self.foundir_client.enabled)
        ocr_layout.addWidget(self.foundir_reconnect_button)
        
        self.ocr_cache_info_button = QPushButton("OCR Cache Info")
        self.ocr_cache_info_button.clicked.connect(self.show_ocr_cache_info)
        ocr_layout.addWidget(self.ocr_cache_info_button)

        # Google Vision comparison toggle
        if self.google_vision_client and self.google_vision_client.enabled:
            self.google_vision_compare_checkbox = QCheckBox("Use Google Vision Comparison")
            self.google_vision_compare_checkbox.setChecked(False)
            self.google_vision_compare_checkbox.toggled.connect(self.toggle_google_vision_comparison)
            self.google_vision_compare_checkbox.setToolTip(
                "Automatically compare Tesseract OCR with Google Vision API when OCR runs.\n"
                "Shows match statistics in status bar.\n\n"
                "Note: Coloring is disabled to avoid conflicts with spell check."
            )
            ocr_layout.addWidget(self.google_vision_compare_checkbox)

        ocr_layout.addStretch()
        main_layout.addWidget(ocr_group)

        # LINE SEGMENTS Controls
        line_seg_group = QGroupBox("Line Segmentation Settings")
        line_seg_layout = QHBoxLayout()
        line_seg_layout.setContentsMargins(10, 5, 10, 5)
        line_seg_group.setLayout(line_seg_layout)

        # Showing line segments
        self.show_line_segments_checkbox = QCheckBox("Show Line Segments")
        self.show_line_segments_checkbox.setChecked(False)
        self.show_line_segments_checkbox.toggled.connect(self.toggle_line_segments_display)
        self.show_line_segments_checkbox.setEnabled(True)
        line_seg_layout.addWidget(self.show_line_segments_checkbox)
        
        self.edit_line_segments_checkbox = QCheckBox("Edit Line Segments")
        self.edit_line_segments_checkbox.setChecked(False)
        self.edit_line_segments_checkbox.toggled.connect(self.toggle_line_segment_editing)
        self.edit_line_segments_checkbox.setEnabled(True)
        line_seg_layout.addWidget(self.edit_line_segments_checkbox)

        self.add_line_segment_checkbox = QCheckBox("Add New Line Segment")
        self.add_line_segment_checkbox.setChecked(False)
        self.add_line_segment_checkbox.toggled.connect(self.toggle_add_line_segment_mode)
        self.add_line_segment_checkbox.setEnabled(True)
        line_seg_layout.addWidget(self.add_line_segment_checkbox)

        self.delete_line_segment_button = QPushButton("Delete Selected Line")
        self.delete_line_segment_button.clicked.connect(self.delete_selected_line_segment)
        self.delete_line_segment_button.setEnabled(True)
        line_seg_layout.addWidget(self.delete_line_segment_button)

        # self.save_line_segments_button = QPushButton("Save Line Segments")
        # self.save_line_segments_button.clicked.connect(self.save_line_segments_to_file)
        # self.save_line_segments_button.setEnabled(True)
        # line_seg_layout.addWidget(self.save_line_segments_button)

        self.load_line_segments_button = QPushButton("Load Line Segments")
        self.load_line_segments_button.clicked.connect(self.load_line_segments_from_file)
        self.load_line_segments_button.setEnabled(True)
        line_seg_layout.addWidget(self.load_line_segments_button)

        line_seg_layout.addStretch()
        main_layout.addWidget(line_seg_group)
        
        # ADD THIS: Word suggestion controls
        self.suggestions_enabled_checkbox = QCheckBox("Enable Word Suggestions")
        self.suggestions_enabled_checkbox.setChecked(True)
        self.suggestions_enabled_checkbox.toggled.connect(self.toggle_suggestions)
        self.suggestions_enabled_checkbox.setToolTip(
            "Enable real-time word suggestions while typing\n"
            "Shortcuts:\n"
            "  • Ctrl+Space: Show suggestions manually\n"
            "  • Ctrl+Shift+C: Suggestions for selected word\n"
            "  • Arrow keys: Navigate suggestions\n"
            "  • Enter: Accept suggestion"
        )
        pdf_layout.addWidget(self.suggestions_enabled_checkbox)
        
        self.suggestion_stats_button = QPushButton("Suggestion Stats")
        self.suggestion_stats_button.clicked.connect(self.show_suggestion_stats)
        pdf_layout.addWidget(self.suggestion_stats_button)

        self.feedback_button = QPushButton("📧 Feedback")
        self.feedback_button.clicked.connect(self.show_feedback_dialog)
        self.feedback_button.setToolTip(
            "Send feedback, report bugs, or request features\n"
            "Your input helps us improve PANDULIPI!"
        )
        self.feedback_button.setStyleSheet("""
            QPushButton {
                background-color: #3498db;
                color: white;
                padding: 5px 15px;
                border: none;
                border-radius: 3px;
                font-weight: bold;
            }
            QPushButton:hover {
                background-color: #2980b9;
            }
            QPushButton:pressed {
                background-color: #21618c;
            }
        """)
        pdf_layout.addWidget(self.feedback_button)
        
        pdf_layout.addStretch()
        main_layout.addWidget(pdf_group)
        
        # Zoom Controls
        zoom_group = QGroupBox("Zoom Controls")
        zoom_layout = QHBoxLayout()
        zoom_layout.setContentsMargins(10, 5, 10, 5)
        zoom_group.setLayout(zoom_layout)
        
        self.zoom_out_button = QPushButton("Zoom Out")
        self.zoom_out_button.clicked.connect(self.zoom_out)
        self.zoom_out_button.setEnabled(False)
        self.zoom_out_button.setToolTip("Zoom Out (Ctrl+-)")
        zoom_layout.addWidget(self.zoom_out_button)
        
        self.zoom_slider = QSlider(Qt.Horizontal)
        self.zoom_slider.setMinimum(MIN_ZOOM)
        self.zoom_slider.setMaximum(MAX_ZOOM)
        self.zoom_slider.setValue(DEFAULT_ZOOM)
        self.zoom_slider.setTickInterval(ZOOM_STEP)
        self.zoom_slider.setTickPosition(QSlider.TicksBelow)
        self.zoom_slider.valueChanged.connect(self.zoom_slider_changed)
        self.zoom_slider.setEnabled(False)
        zoom_layout.addWidget(self.zoom_slider, 1)
        
        self.zoom_in_button = QPushButton("Zoom In")
        self.zoom_in_button.clicked.connect(self.zoom_in)
        self.zoom_in_button.setEnabled(False)
        self.zoom_in_button.setToolTip("Zoom In (Ctrl++)")
        zoom_layout.addWidget(self.zoom_in_button)
        
        self.zoom_label = QLabel(f"{DEFAULT_ZOOM}%")
        zoom_layout.addWidget(self.zoom_label)
        
        self.zoom_reset_button = QPushButton("Reset Zoom")
        self.zoom_reset_button.clicked.connect(self.zoom_reset)
        self.zoom_reset_button.setEnabled(False)
        self.zoom_reset_button.setToolTip("Reset Zoom (Ctrl+0)")
        zoom_layout.addWidget(self.zoom_reset_button)

        # Keyboard shortcuts
        self.zoom_in_shortcut = QShortcut(QKeySequence("Ctrl++"), self)
        self.zoom_in_shortcut.activated.connect(self.zoom_in)
        self.zoom_out_shortcut = QShortcut(QKeySequence("Ctrl+-"), self)
        self.zoom_out_shortcut.activated.connect(self.zoom_out)
        self.zoom_reset_shortcut = QShortcut(QKeySequence("Ctrl+0"), self)
        self.zoom_reset_shortcut.activated.connect(self.zoom_reset)
        
        main_layout.addWidget(zoom_group)
        
        # Selection Controls
        selection_group = QGroupBox("Akshara Selection")
        selection_layout = QHBoxLayout()
        selection_layout.setContentsMargins(10, 5, 10, 5)
        selection_group.setLayout(selection_layout)
        
        self.selection_checkbox = QCheckBox("Enable Manual Selection")
        self.selection_checkbox.setChecked(False)
        self.selection_checkbox.toggled.connect(self.toggle_selection_mode)
        self.selection_checkbox.setEnabled(False)
        selection_layout.addWidget(self.selection_checkbox)
        
        self.add_selection_button = QPushButton("Add Selection")
        self.add_selection_button.clicked.connect(self.add_selection)
        self.add_selection_button.setEnabled(False)
        selection_layout.addWidget(self.add_selection_button)
        
        self.save_selections_button = QPushButton("Save Selections")
        self.save_selections_button.clicked.connect(self.save_selections_to_images)
        self.save_selections_button.setEnabled(False)
        selection_layout.addWidget(self.save_selections_button)
        
        self.clear_selection_button = QPushButton("Clear Selections")
        self.clear_selection_button.clicked.connect(self.clear_selection)
        self.clear_selection_button.setEnabled(False)
        selection_layout.addWidget(self.clear_selection_button)

        self.clear_page_selections_button = QPushButton("Clear Page Selections")
        self.clear_page_selections_button.clicked.connect(self.clear_all_selections_on_page)
        self.clear_page_selections_button.setEnabled(False)
        selection_layout.addWidget(self.clear_page_selections_button)
        
        self.load_text_button = QPushButton("Load Text File")
        self.load_text_button.clicked.connect(self.load_text_to_editor)
        self.load_text_button.setEnabled(False)
        selection_layout.addWidget(self.load_text_button)

        self.save_text_button = QPushButton("Save Text")
        self.save_text_button.clicked.connect(self.save_text_from_editor)
        self.save_text_button.setEnabled(False)
        selection_layout.addWidget(self.save_text_button)
            
        selection_layout.addStretch()
        main_layout.addWidget(selection_group)
        
        # Splitter for PDF view and text
        splitter = QSplitter(Qt.Horizontal)
        
        # PDF view
        self.pdf_scroll_area = QScrollArea()
        self.pdf_scroll_area.setWidgetResizable(True)
        self.pdf_content = PDFViewLabel()
        self.pdf_content.setText("Load a PDF to begin")
        self.pdf_content.setAlignment(Qt.AlignCenter)
        self.pdf_content.selectionChanged.connect(self.update_selection)
        self.pdf_content.selectionFinished.connect(self.finalize_selection)
        self.pdf_scroll_area.setWidget(self.pdf_content)
        splitter.addWidget(self.pdf_scroll_area)

        # Text editor
        self.text_edit = QTextEdit()
        self.text_edit.setReadOnly(False)
        self.text_edit.setPlaceholderText("Text editor - Type or use speech recognition")
        splitter.addWidget(self.text_edit)

        # Initialize spell check highlighter
        self.highlighter = SpellCheckHighlighter(
            self.text_edit.document(),
            self.spell_checker
        )
        self.highlighter.enable(False)

        # ═══════════════════════════════════════════════════════════════
        # ADD THIS: Initialize word suggestions
        # ═══════════════════════════════════════════════════════════════
        if self.suggestion_manager.enabled:
            add_suggestions_to_text_edit(self.text_edit, self.suggestion_manager)
            print("✓ Word suggestions initialized")
        else:
            print("⚠ Word suggestions not available (GBook dictionary not found)")

        # Set font
        hindi_font = QFont()
        hindi_font.setPointSize(16)
        hindi_font.setFamily("Noto Sans Devanagari") 
        self.text_edit.setFont(hindi_font)
        self.text_edit.setAttribute(Qt.WA_InputMethodEnabled, True)
        
        splitter.setSizes([800, 600])
        main_layout.addWidget(splitter, 1)
        
        # Speech recognition UI
        self.add_speech_recognition_ui()
        
        # Typing settings UI
        self.add_typing_settings_ui()

        # Status bar
        self.statusBar().showMessage("Ready - Load a PDF to begin annotation")

    # ═══════════════════════════════════════════════════════════════════
    # FEEDBACK SETTINGS
    # ═══════════════════════════════════════════════════════════════════
    def show_feedback_dialog(self):
        """Show feedback dialog with application context"""
        try:
            # Gather application context for feedback
            app_context = {
                'app_version': '1.0.0',  # Update with your actual version
                'document_type': getattr(self, 'current_document_type', 'None'),
                'current_page': str(self.current_page + 1) if hasattr(self, 'current_page') else 'N/A',
                'total_pages': str(self.total_pages) if hasattr(self, 'total_pages') else 'N/A',
                'ocr_language': getattr(self, 'ocr_language', 'Unknown'),
                'foundir_enabled': getattr(self, 'foundir_enabled', False),
                'spell_check_enabled': (self.spell_check_enabled_checkbox.isChecked() 
                                      if hasattr(self, 'spell_check_enabled_checkbox') else False),
                'current_file': getattr(self, 'current_pdf_filename', 'None'),
            }
            
            # Create and show feedback dialog
            dialog = FeedbackDialog(self, app_context)
            result = dialog.exec_()
            
            if result == QDialog.Accepted:
                self.statusBar().showMessage("Thank you for your feedback!", 5000)
            
        except Exception as e:
            QMessageBox.warning(
                self,
                "Feedback Error",
                f"Could not open feedback dialog:\n{str(e)}\n\n"
                "Please ensure feedback_module.py is in the same directory."
            )
            print(f"Error opening feedback dialog: {e}")
            import traceback
            traceback.print_exc()

    # ═══════════════════════════════════════════════════════════════════
    # TYPING SETTINGS 
    # ═══════════════════════════════════════════════════════════════════
    def add_typing_settings_ui(self):
        """Add typing settings UI"""
        typing_group = QGroupBox("Typing Settings")
        typing_layout = QHBoxLayout()
        typing_layout.setContentsMargins(10, 5, 10, 5)
        typing_group.setLayout(typing_layout)

        typing_layout.addWidget(QLabel("Input Method:"))
        self.keyboard_combo = QComboBox()
        self.keyboard_combo.addItems(["English", "Hindi"])
        self.keyboard_combo.currentIndexChanged.connect(self.change_keyboard_layout)
        typing_layout.addWidget(self.keyboard_combo)

        self.hindi_help_button = QPushButton("Hindi Typing Help")
        self.hindi_help_button.clicked.connect(self.show_hindi_typing_help)
        typing_layout.addWidget(self.hindi_help_button)
        
        typing_layout.addStretch()
        self.main_layout.addWidget(typing_group)

    # ═══════════════════════════════════════════════════════════════════
    # ADDING THE SUGGESTIONS  
    # ═══════════════════════════════════════════════════════════════════
    def toggle_suggestions(self, enabled):
        """Toggle word suggestions on/off"""
        self.suggestions_enabled = enabled
        
        if hasattr(self.text_edit, 'enable_suggestions'):
            self.text_edit.enable_suggestions(enabled)
        
        status = "enabled" if enabled else "disabled"
        self.statusBar().showMessage(f"Word suggestions {status}")

    def show_suggestion_stats(self):
        """Show word suggestion statistics"""
        if not self.suggestion_manager.enabled:
            QMessageBox.information(
                self,
                "Suggestions Not Available",
                "Word suggestion engine is not initialized.\n"
                "Make sure GBook dictionary exists in Dictionaries folder."
            )
            return
        
        # Gather statistics
        stats_lines = ["Word Suggestion Statistics", "=" * 40, ""]
        
        for name, engine in self.suggestion_manager.engines.items():
            stats_lines.append(f"{name.upper()} Dictionary:")
            stats_lines.append(f"  Total words: {engine.word_count:,}")
            stats_lines.append(f"  Status: {'✓ Active' if engine.enabled else '✗ Inactive'}")
            stats_lines.append("")
        
        stats_lines.append("Features:")
        stats_lines.append("  • Real-time suggestions while typing")
        stats_lines.append("  • Fuzzy matching for typos")
        stats_lines.append("  • Prefix-based word completion")
        stats_lines.append("")
        stats_lines.append("Keyboard Shortcuts:")
        stats_lines.append("  • Ctrl+Space: Show suggestions")
        stats_lines.append("  • Ctrl+Shift+C: Suggest for selected word")
        stats_lines.append("  • Up/Down: Navigate suggestions")
        stats_lines.append("  • Enter: Accept suggestion")
        stats_lines.append("  • Esc: Close suggestions")
        
        stats_text = "\n".join(stats_lines)
        
        msg_box = QMessageBox(self)
        msg_box.setWindowTitle("Word Suggestion Statistics")
        msg_box.setText(stats_text)
        msg_box.setFont(QFont("Courier New", 10))
        msg_box.setStandardButtons(QMessageBox.Ok)
        msg_box.exec_()

    # ═══════════════════════════════════════════════════════════════════
    # Line Segments Opertions on Uploaded Image/PDF
    # ═══════════════════════════════════════════════════════════════════
    def detect_line_segments_for_current_page(self):
        """Detect line segments for current page without running full OCR"""
        if not self.curr_page_pixmap:
            return []
        
        try:
            # Convert pixmap to image
            image = self.curr_page_pixmap.toImage()
            arr = qimage_to_numpy(image)
            
            # Convert to grayscale
            gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            
            # Apply adaptive thresholding
            image_processor = ImageProcessor()
            binary = image_processor.apply_adaptive_thresholding(gray)
            binary = image_processor.ensure_black_text_on_white(binary)
            
            # Perform line segmentation
            line_segments = image_processor.segment_lines(binary)
            
            # Convert to QRect format
            line_segment_rects = []
            for start_row, end_row in line_segments:
                rect = QRect(0, start_row, binary.shape[1], end_row - start_row)
                line_segment_rects.append(rect)
            
            print(f"Detected {len(line_segment_rects)} line segments for page {self.current_page + 1}")
            return line_segment_rects
            
        except Exception as e:
            print(f"Error detecting line segments: {e}")
            return []

    def toggle_line_segment_editing(self, enabled):
        """Toggle line segment editing mode"""
        if enabled:
            # Disable other modes
            self.selection_checkbox.setChecked(False)
            self.add_line_segment_checkbox.setChecked(False)
            
            # Check if line segments exist for current page
            if self.current_page not in self.line_segments_cache:
                # Auto-detect line segments
                reply = QMessageBox.question(
                    self,
                    "No Line Segments",
                    "No line segments available for this page.\n\n"
                    "Would you like to auto-detect line segments now?",
                    QMessageBox.Yes | QMessageBox.No,
                    QMessageBox.Yes
                )
                
                if reply == QMessageBox.Yes:
                    self.statusBar().showMessage(
                        f"Detecting line segments for page {self.current_page + 1}..."
                    )
                    
                    line_segments = self.detect_line_segments_for_current_page()
                    
                    if line_segments:
                        self.line_segments_cache[self.current_page] = line_segments
                        # Auto-enable display
                        self.show_line_segments_checkbox.setChecked(True)
                        self.show_line_segments_checkbox.setEnabled(True)
                        self.display_line_segments(line_segments)
                    else:
                        QMessageBox.warning(
                            self,
                            "Detection Failed",
                            "Could not detect line segments for this page."
                        )
                        self.edit_line_segments_checkbox.setChecked(False)
                        return
                else:
                    self.edit_line_segments_checkbox.setChecked(False)
                    return
            
            # Enable editing mode
            if hasattr(self.pdf_content, 'enable_line_segment_editing'):
                self.pdf_content.enable_line_segment_editing(True)
                self.pdf_content.lineSegmentChanged.connect(self.handle_line_segment_changed)
            
            self.line_segment_editing_mode = True
            # self.save_line_segments_button.setEnabled(True)
            self.delete_line_segment_button.setEnabled(True)
            self.statusBar().showMessage(
                "Line segment editing enabled - Click and drag edges/corners to resize, "
                "or drag center to move. Click a line to select it for deletion."
            )
        else:
            # Disable editing mode
            if hasattr(self.pdf_content, 'enable_line_segment_editing'):
                self.pdf_content.enable_line_segment_editing(False)
                try:
                    self.pdf_content.lineSegmentChanged.disconnect(self.handle_line_segment_changed)
                except:
                    pass
            
            self.line_segment_editing_mode = False
            # self.save_line_segments_button.setEnabled(False)
            self.delete_line_segment_button.setEnabled(False)
            self.statusBar().showMessage("Line segment editing disabled")

    def handle_line_segment_changed(self, index, rect):
        """Handle line segment modification"""
        # Convert from display coordinates to original coordinates (at default zoom)
        original_rect = QRect(
            int(rect.x() / self.zoom_factor),
            int(rect.y() / self.zoom_factor),
            int(rect.width() / self.zoom_factor),
            int(rect.height() / self.zoom_factor)
        )
        
        # Update cache with original coordinates
        if self.current_page in self.line_segments_cache:
            self.line_segments_cache[self.current_page][index] = original_rect
            # Enable save button since segments were modified
            # self.save_line_segments_button.setEnabled(True)
        
        self.statusBar().showMessage(
            f"Line segment L{index+1} updated: "
            f"({original_rect.x()}, {original_rect.y()}, "
            f"{original_rect.width()}x{original_rect.height()}) - Click 'Save' to persist changes"
        )

    def save_line_segments_to_file(self):
        """Save line segments to JSON file"""
        if self.current_page not in self.line_segments_cache:
            QMessageBox.warning(self, "No Segments", "No line segments to save for current page.")
            return
        
        if not self.current_pdf_filename:
            QMessageBox.warning(self, "No PDF", "No PDF file loaded.")
            return
        
        try:
            # Create filename based on PDF name and page number
            pdf_base = os.path.splitext(self.current_pdf_filename)[0]
            segments_file = os.path.join(
                self.line_segments_dir,
                f"{pdf_base}_page{self.current_page + 1}_segments.json"
            )
            
            # Convert QRect objects to serializable format
            segments_data = {
                'pdf_filename': self.current_pdf_filename,
                'page_number': self.current_page + 1,
                'total_pages': self.total_pages,
                'default_zoom': DEFAULT_ZOOM,
                'line_segments': [
                    {
                        'index': i,
                        'x': rect.x(),
                        'y': rect.y(),
                        'width': rect.width(),
                        'height': rect.height()
                    }
                    for i, rect in enumerate(self.line_segments_cache[self.current_page])
                ],
                'timestamp': get_timestamp()
            }
            
            # Atomic write
            temp_file = segments_file + '.tmp'
            with open(temp_file, 'w', encoding='utf-8') as f:
                json.dump(segments_data, f, indent=4, ensure_ascii=False)
            
            if not os.path.exists(temp_file):
                raise IOError("Temporary file not created")
            
            os.replace(temp_file, segments_file)
            
            # Clear OCR cache for this page to force re-processing with new segments
            if self.current_page in self.ocr_cache:
                del self.ocr_cache[self.current_page]
                print(f"Cleared OCR cache for page {self.current_page + 1} - will use new segments on next OCR")
            
            # Disable save button until next modification
            # self.save_line_segments_button.setEnabled(False)
            
            self.statusBar().showMessage(
                f"Saved {len(self.line_segments_cache[self.current_page])} line segments to "
                f"{os.path.basename(segments_file)}"
            )
            
            # QMessageBox.information(
            #     self,
            #     "Segments Saved",
            #     f"Line segments saved successfully:\n{os.path.basename(segments_file)}\n\n"
            #     f"Total segments: {len(self.line_segments_cache[self.current_page])}\n\n"
            #     f"✓ These segments will be used for OCR processing\n"
            #     f"✓ Each line will be denoised with FoundIR (if enabled)\n"
            #     f"✓ Click 'Run OCR Now' to process with saved segments"
            # )
            
        except Exception as e:
            QMessageBox.critical(
                self,
                "Save Error",
                f"Could not save line segments:\n{str(e)}"
            )
            print(f"Error saving line segments: {e}")

    def load_line_segments_from_file(self):
        """Load line segments from JSON file"""
        if not self.current_pdf_filename:
            QMessageBox.warning(self, "No PDF", "Please load a PDF file first.")
            return
        
        # Default filename
        pdf_base = os.path.splitext(self.current_pdf_filename)[0]
        default_file = os.path.join(
            self.line_segments_dir,
            f"{pdf_base}_page{self.current_page + 1}_segments.json"
        )
        
        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Load Line Segments",
            default_file,
            "JSON Files (*.json);;All Files (*.*)"
        )
        
        if not file_path:
            return
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                segments_data = json.load(f)
            
            # Validate data
            if not isinstance(segments_data, dict):
                raise ValueError("Invalid JSON structure")
            
            if 'line_segments' not in segments_data:
                raise ValueError("Missing 'line_segments' key")
            
            # Verify it's for the current PDF and page
            pdf_filename = segments_data.get('pdf_filename', '')
            page_number = segments_data.get('page_number', 0)
            
            if pdf_filename != self.current_pdf_filename:
                reply = QMessageBox.question(
                    self,
                    "Different PDF",
                    f"Line segments are from '{pdf_filename}',\n"
                    f"but current PDF is '{self.current_pdf_filename}'.\n\n"
                    f"Load anyway?",
                    QMessageBox.Yes | QMessageBox.No,
                    QMessageBox.No
                )
                if reply == QMessageBox.No:
                    return
            
            if page_number != self.current_page + 1:
                reply = QMessageBox.question(
                    self,
                    "Different Page",
                    f"Line segments are from page {page_number},\n"
                    f"but current page is {self.current_page + 1}.\n\n"
                    f"Load anyway?",
                    QMessageBox.Yes | QMessageBox.No,
                    QMessageBox.No
                )
                if reply == QMessageBox.No:
                    return
            
            # Convert from JSON to QRect objects
            line_segments = []
            for seg_data in segments_data['line_segments']:
                rect = QRect(
                    seg_data['x'],
                    seg_data['y'],
                    seg_data['width'],
                    seg_data['height']
                )
                line_segments.append(rect)
            
            # Store in cache
            self.line_segments_cache[self.current_page] = line_segments
            
            # Clear OCR cache for this page to force re-processing
            if self.current_page in self.ocr_cache:
                del self.ocr_cache[self.current_page]
                print(f"Cleared OCR cache for page {self.current_page + 1}")
            
            # Display
            self.show_line_segments_checkbox.setChecked(True)
            self.show_line_segments_checkbox.setEnabled(True)
            self.edit_line_segments_checkbox.setEnabled(True)
            self.add_line_segment_checkbox.setEnabled(True)
            self.display_line_segments(line_segments)
            
            self.statusBar().showMessage(
                f"Loaded {len(line_segments)} line segments from "
                f"{os.path.basename(file_path)}"
            )
            
            QMessageBox.information(
                self,
                "Segments Loaded",
                f"Successfully loaded line segments:\n{os.path.basename(file_path)}\n\n"
                f"Total segments: {len(line_segments)}\n\n"
                f"✓ These segments will be used for OCR processing\n"
                f"✓ OCR cache cleared - ready for re-processing\n"
                f"✓ Click 'Run OCR Now' to process with loaded segments"
            )
            
        except Exception as e:
            QMessageBox.critical(
                self,
                "Load Error",
                f"Could not load line segments:\n{str(e)}"
            )
            print(f"Error loading line segments: {e}")

    def toggle_add_line_segment_mode(self, enabled):
        """Toggle add line segment mode"""
        if enabled:
            # Disable other modes
            self.selection_checkbox.setChecked(False)
            self.edit_line_segments_checkbox.setChecked(False)
            
            # Check if we can add line segments
            if not self.curr_page_pixmap:
                QMessageBox.warning(
                    self,
                    "No Page",
                    "Please load a PDF page first."
                )
                self.add_line_segment_checkbox.setChecked(False)
                return
            
            # Initialize line segments cache if not exists
            if self.current_page not in self.line_segments_cache:
                # Ask if user wants to auto-detect first
                reply = QMessageBox.question(
                    self,
                    "No Line Segments",
                    "No line segments detected yet.\n\n"
                    "Would you like to auto-detect line segments first?\n"
                    "(You can then add more manually)",
                    QMessageBox.Yes | QMessageBox.No,
                    QMessageBox.No
                )
                
                if reply == QMessageBox.Yes:
                    self.statusBar().showMessage(
                        f"Detecting line segments for page {self.current_page + 1}..."
                    )
                    
                    line_segments = self.detect_line_segments_for_current_page()
                    
                    if line_segments:
                        self.line_segments_cache[self.current_page] = line_segments
                        self.display_line_segments(line_segments)
                    else:
                        # Start with empty list
                        self.line_segments_cache[self.current_page] = []
                else:
                    # Start with empty list
                    self.line_segments_cache[self.current_page] = []
                
                # Enable display
                self.show_line_segments_checkbox.setChecked(True)
                self.show_line_segments_checkbox.setEnabled(True)
            
            # Enable add mode
            if hasattr(self.pdf_content, 'enable_add_line_segment'):
                self.pdf_content.enable_add_line_segment(True)
                self.pdf_content.newLineSegmentCreated.connect(self.handle_new_line_segment_created)
            
            self.add_line_segment_mode = True
            # self.save_line_segments_button.setEnabled(True)
            self.statusBar().showMessage(
                "Add line segment mode enabled - Click and drag to draw new line segment box"
            )
        else:
            # Disable add mode
            if hasattr(self.pdf_content, 'enable_add_line_segment'):
                self.pdf_content.enable_add_line_segment(False)
                try:
                    self.pdf_content.newLineSegmentCreated.disconnect(self.handle_new_line_segment_created)
                except:
                    pass
            
            self.add_line_segment_mode = False
            self.statusBar().showMessage("Add line segment mode disabled")

    def handle_new_line_segment_created(self, rect):
        """Handle new line segment creation"""
        # Convert from display coordinates to original coordinates (at default zoom)
        original_rect = QRect(
            int(rect.x() / self.zoom_factor),
            int(rect.y() / self.zoom_factor),
            int(rect.width() / self.zoom_factor),
            int(rect.height() / self.zoom_factor)
        )
        
        # Add to cache
        if self.current_page not in self.line_segments_cache:
            self.line_segments_cache[self.current_page] = []
        
        self.line_segments_cache[self.current_page].append(original_rect)
        
        # Add to display
        if hasattr(self.pdf_content, 'add_new_line_segment'):
            self.pdf_content.add_new_line_segment(rect)
        
        # Enable controls
        self.edit_line_segments_checkbox.setEnabled(True)
        self.delete_line_segment_button.setEnabled(True)
        # self.save_line_segments_button.setEnabled(True)  # Enable save
        
        num_segments = len(self.line_segments_cache[self.current_page])
        self.statusBar().showMessage(
            f"New line segment L{num_segments} created: "
            f"({original_rect.x()}, {original_rect.y()}, "
            f"{original_rect.width()}x{original_rect.height()}) - Click 'Save' to persist"
        )

    def delete_selected_line_segment(self):
        """Delete the currently selected line segment"""
        if not hasattr(self.pdf_content, 'selected_line_index'):
            return
        
        selected_index = self.pdf_content.selected_line_index
        
        if selected_index < 0:
            QMessageBox.warning(
                self,
                "No Selection",
                "Please select a line segment to delete first.\n"
                "Enable 'Edit Line Segments' mode and click on a line to select it."
            )
            return
        
        if self.current_page not in self.line_segments_cache:
            return
        
        if selected_index >= len(self.line_segments_cache[self.current_page]):
            return
        
        # Confirm deletion
        reply = QMessageBox.question(
            self,
            "Confirm Deletion",
            f"Delete line segment L{selected_index + 1}?",
            QMessageBox.Yes | QMessageBox.No,
            QMessageBox.No
        )
        
        if reply == QMessageBox.Yes:
            # Delete from cache
            del self.line_segments_cache[self.current_page][selected_index]
            
            # Delete from display
            if hasattr(self.pdf_content, 'delete_line_segment'):
                self.pdf_content.delete_line_segment(selected_index)
            
            # Disable delete button if no segments left
            if len(self.line_segments_cache[self.current_page]) == 0:
                self.delete_line_segment_button.setEnabled(False)
                self.edit_line_segments_checkbox.setEnabled(False)
            
            self.statusBar().showMessage(
                f"Deleted line segment L{selected_index + 1}. "
                f"Remaining: {len(self.line_segments_cache[self.current_page])}"
            )

    ### New methods for adding line segments in the image
    def toggle_line_segments_display(self, enabled):
        """Toggle line segment visualization"""
        if enabled:
            # Check if line segments exist for current page
            if self.current_page not in self.line_segments_cache:
                # Auto-detect line segments
                self.statusBar().showMessage(
                    f"Detecting line segments for page {self.current_page + 1}..."
                )
                
                line_segments = self.detect_line_segments_for_current_page()
                
                if line_segments:
                    self.line_segments_cache[self.current_page] = line_segments
                    # Enable editing controls
                    self.edit_line_segments_checkbox.setEnabled(True)
                    self.add_line_segment_checkbox.setEnabled(True)
                    # self.save_line_segments_button.setEnabled(False)  # Not saved yet
                    self.load_line_segments_button.setEnabled(True)
                else:
                    self.statusBar().showMessage("No line segments detected for this page")
                    self.show_line_segments_checkbox.setChecked(False)
                    return
            
            # Display line segments for current page
            if self.current_page in self.line_segments_cache:
                self.display_line_segments(self.line_segments_cache[self.current_page])
                self.statusBar().showMessage(
                    f"Showing {len(self.line_segments_cache[self.current_page])} line segments"
                )
            else:
                self.statusBar().showMessage("No line segments available for this page")
        else:
            # Clear line segments
            if hasattr(self.pdf_content, 'clear_line_segments'):
                self.pdf_content.clear_line_segments()
            self.statusBar().showMessage("Line segments hidden")

    def display_line_segments(self, line_segments):
        """Display line segments on the PDF image"""
        if not line_segments or not hasattr(self.pdf_content, 'set_line_segments'):
            return
        
        # Scale line segments to current zoom
        scaled_segments = []
        for rect in line_segments:
            scaled_rect = QRect(
                int(rect.x() * self.zoom_factor),
                int(rect.y() * self.zoom_factor),
                int(rect.width() * self.zoom_factor),
                int(rect.height() * self.zoom_factor)
            )
            scaled_segments.append(scaled_rect)
        
        self.pdf_content.set_line_segments(scaled_segments)
        
    def load_cached_ocr_text(self, page_num):
        """Load OCR text from cache with optional Google Vision comparison"""
        text = self.ocr_cache.get(page_num, "")
        
        if text.strip():
            # Check if Google Vision comparison should be applied
            if (self.use_google_vision_comparison and 
                page_num in self.google_vision_cache):
                # Apply comparison highlighting
                google_text = self.google_vision_cache[page_num]
                self.apply_comparison_highlighting(text, google_text)
            else:
                # Normal text loading
                self.text_edit.clear()
                cursor = self.text_edit.textCursor()
                cursor.movePosition(cursor.Start)
                cursor.insertText(text)
                self.text_edit.setTextCursor(cursor)
        else:
            self.text_edit.clear()

    def update_ocr_progress(self, value):
        """Update OCR progress bar"""
        self.ocr_progress.setValue(value)

    def handle_ocr_error(self, error_message):
        """Handle OCR errors"""
        self.ocr_in_progress.clear()
        self.statusBar().showMessage(error_message)
        self.ocr_progress.setVisible(False)
        QMessageBox.warning(self, "OCR Error", error_message)

    def show_ocr_cache_info(self):
        """Show OCR cache information"""
        total_pages = self.total_pages
        cached_pages = len(self.ocr_cache)
        in_progress = len(self.ocr_in_progress)
        
        cached_list = sorted(self.ocr_cache.keys())
        cached_str = ', '.join([str(p+1) for p in cached_list]) if cached_list else "None"
        
        info_text = f"""OCR Cache Information:
        
Total Pages: {total_pages}
Cached Pages: {cached_pages}
In Progress: {in_progress}

Cached page numbers: {cached_str}
"""
        
        QMessageBox.information(self, "OCR Cache Info", info_text)

    # ═══════════════════════════════════════════════════════════════════
    # MULTI-OCR CONFIDENCE: Google Vision API
    # ═══════════════════════════════════════════════════════════════════
    def init_google_vision_ocr(self):
        """Initialize Google Vision API client"""
        try:
            credentials_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
            # Or use explicit path:
            # credentials_path = "./pandulipi-482118.json"
            
            self.google_vision_client = GoogleVisionOCR(credentials_path)
            
        except Exception as e:
            print(f"⚠ Could not initialize Google Vision API: {e}")
            self.google_vision_client = None

    def perform_google_vision_ocr(self, pixmap, page_num):
        """
        Perform OCR using Google Vision API with caching
        
        Args:
            pixmap: QPixmap of the page
            page_num: Page number
        """
        if not self.google_vision_client or not self.google_vision_client.enabled:
            self.statusBar().showMessage("Google Vision API not available")
            return
        
        # Check cache first
        if page_num in self.ocr_cache:
            self.load_cached_ocr_text(page_num)
            self.statusBar().showMessage(f"Loaded cached OCR for page {page_num + 1}")
            return
        
        # Check if already processing
        if page_num in self.ocr_in_progress:
            self.statusBar().showMessage(f"OCR already in progress for page {page_num + 1}")
            return
        
        self.ocr_in_progress.add(page_num)
        self.ocr_progress.setVisible(True)
        self.ocr_progress.setValue(0)
        
        # Language hints for Google Vision
        language_hints = ['hi', 'sa', 'en']  # Hindi, Sanskrit, English
        
        self.statusBar().showMessage(
            f"Running Google Vision OCR on page {page_num + 1}..."
        )
        
        # Create and start worker
        worker = GoogleVisionOCRWorker(
            pixmap,
            page_num,
            self.google_vision_client,
            language_hints,
            self.spell_checker,
            self
        )
        worker.result_ready.connect(self.handle_ocr_result)
        worker.progress_update.connect(self.update_ocr_progress)
        worker.error_occurred.connect(self.handle_ocr_error)
        self.ocr_workers.append(worker)
        worker.start()

    def init_text_editor_context_menu(self):
        """Initialize custom context menu"""
        self.text_edit.setContextMenuPolicy(Qt.CustomContextMenu)
        self.text_edit.customContextMenuRequested.connect(self.show_text_editor_context_menu)

    def toggle_ocr_engine(self, use_google_vision):
        """Toggle between Google Vision and Tesseract OCR"""
        self.use_google_vision = use_google_vision
        
        if use_google_vision and self.google_vision_client and self.google_vision_client.enabled:
            self.statusBar().showMessage("Switched to Google Vision OCR")
        else:
            self.use_google_vision = False
            self.statusBar().showMessage("Using Tesseract OCR")

    def show_text_editor_context_menu(self, position):
        """Show custom context menu"""
        menu = self.text_edit.createStandardContextMenu(position)
        menu.addSeparator()
        
        hindi_help_action = menu.addAction("Hindi Typing Help")
        hindi_help_action.triggered.connect(self.show_hindi_typing_help)
        
        menu.exec_(self.text_edit.mapToGlobal(position))

    def toggle_google_vision_comparison(self, enabled):
        """Toggle Google Vision comparison mode"""
        self.use_google_vision_comparison = enabled
        
        if enabled:
            # Disable spell check when using Google Vision comparison
            self.spell_check_enabled_checkbox.setChecked(False)
            self.spell_check_enabled_checkbox.setEnabled(False)
            self.statusBar().showMessage("Google Vision comparison mode enabled - Spell check disabled")
        else:
            # Re-enable spell check checkbox
            self.spell_check_enabled_checkbox.setEnabled(True)
            self.statusBar().showMessage("Google Vision comparison mode disabled")
        
        # If we already have cached results for current page, apply comparison now
        if enabled and self.current_page in self.ocr_cache and self.current_page in self.google_vision_cache:
            tesseract_text = self.ocr_cache[self.current_page]
            google_text = self.google_vision_cache[self.current_page]
            self.apply_comparison_highlighting(tesseract_text, google_text)

    def handle_google_vision_result(self, google_text, page_num):
        """
        Handle Google Vision OCR result and compare with Tesseract.
        Apply color coding based on matches when spell check is disabled.
        """
        # Store in cache
        self.google_vision_cache[page_num] = google_text
        
        print(f"Google Vision OCR completed for page {page_num + 1}: {len(google_text)} characters")
        
        # Only process if this is the current page and comparison is enabled
        if page_num != self.current_page:
            print(f"  Cached for later (current page: {self.current_page + 1})")
            return
        
        if not self.use_google_vision_comparison:
            print(f"  Comparison mode disabled, ignoring")
            return
        
        # Get Tesseract OCR text from cache
        tesseract_text = self.ocr_cache.get(page_num, "")
        
        if not tesseract_text:
            print(f"  Waiting for Tesseract OCR to complete...")
            return
        
        # Both results are ready - apply comparison highlighting
        print(f"  Applying comparison highlighting...")
        self.apply_comparison_highlighting(tesseract_text, google_text)
        
        self.statusBar().showMessage(
            f"Google Vision comparison complete for page {page_num + 1} "
            f"(Tesseract: {len(tesseract_text)} chars, Google: {len(google_text)} chars)"
        )

    def apply_comparison_highlighting(self, tesseract_text, google_vision_text):
        """
        Compare Tesseract and Google Vision texts.
        DISABLED: Just load plain text without coloring to avoid conflicts with spell check.
        """
        # Just load the Tesseract text as plain text
        self.text_edit.clear()
        cursor = self.text_edit.textCursor()
        cursor.insertText(tesseract_text)
        
        # Calculate statistics for status bar (without coloring)
        tesseract_words = self.extract_words(tesseract_text)
        google_words = self.extract_words(google_vision_text)
        google_words_set = set(google_words)
        
        exact_matches = [w for w in tesseract_words if w in google_words_set]
        total_words = len(tesseract_words)
        match_rate = len(exact_matches) / total_words * 100 if total_words else 0
        
        # Update status bar with comparison stats (no coloring)
        self.statusBar().showMessage(
            f"Google Vision comparison: {len(exact_matches)}/{total_words} words matched "
            f"({match_rate:.1f}%) - Coloring disabled to avoid spell check conflicts"
        )

    def extract_words(self, text):
        """Extract words from text (Devanagari and ASCII)"""
        pattern = r'[\u0900-\u097F]+|[a-zA-Z]+'
        return [match.group() for match in re.finditer(pattern, text)]

    def extract_all_subwords(self, words, min_length=3):
        """Extract all possible subwords from word list"""
        subwords = set()
        
        for word in words:
            word_len = len(word)
            # Extract all subwords of length >= min_length
            for length in range(min_length, word_len + 1):
                for start in range(word_len - length + 1):
                    subword = word[start:start + length]
                    subwords.add(subword)
        
        return subwords

    def find_subword_matches_in_set(self, word, subword_set):
        """
        Find all subwords of 'word' that exist in subword_set.
        Returns list of (start, length) tuples for matched subwords.
        """
        matches = []
        word_len = len(word)
        
        # Try all possible subword lengths (longest first)
        for length in range(word_len, 2, -1):  # min length 3
            for start in range(word_len - length + 1):
                subword = word[start:start + length]
                
                # Check if this position is already covered
                if self.is_position_covered_by_matches(matches, start, length):
                    continue
                
                if subword in subword_set:
                    matches.append((start, length))
        
        return sorted(matches, key=lambda x: x[0])  # Sort by position

    def is_position_covered_by_matches(self, matches, start, length):
        """Check if position is already covered by existing matches"""
        end = start + length
        
        for match_start, match_len in matches:
            match_end = match_start + match_len
            # Check for overlap
            if not (end <= match_start or start >= match_end):
                return True
        
        return False

    def insert_word_with_subword_colors(self, cursor, word, subword_matches):
        """
        Insert word with subword highlighting.
        Matched subwords: Gray
        Unmatched parts: Black
        """
        # Create format for matched (gray) and unmatched (black)
        gray_fmt = QTextCharFormat()
        gray_fmt.setForeground(QColor(128, 128, 128))  # Gray
        
        black_fmt = QTextCharFormat()
        black_fmt.setForeground(QColor(0, 0, 0))  # Black
        
        # Track covered positions
        covered = set()
        for start, length in subword_matches:
            for i in range(start, start + length):
                covered.add(i)
        
        # Insert character by character with appropriate color
        for i, char in enumerate(word):
            if i in covered:
                cursor.insertText(char, gray_fmt)
            else:
                cursor.insertText(char, black_fmt)

    # ═══════════════════════════════════════════════════════════════════
    # SPELL CHECK METHODS
    # ═══════════════════════════════════════════════════════════════════

    def toggle_spell_check(self, enabled):
        """Toggle spell check highlighting on/off"""
        if self.highlighter:
            self.highlighter.enable(enabled)
            status = "enabled" if enabled else "disabled"
            self.statusBar().showMessage(f"Spell check {status}")
            
            # Update button states
            self.show_legend_button.setEnabled(enabled)
            self.dict_stats_button.setEnabled(enabled)
            self.add_to_dict_button.setEnabled(enabled)
            
            # ═══════════════════════════════════════════════════════════
            # ADD THIS: Suggestions can work alongside spell check
            # ═══════════════════════════════════════════════════════════
            # Note: Unlike Google Vision, suggestions don't interfere with spell check
            # Both can be enabled simultaneously
            
            # Disable Google Vision comparison when spell check is enabled (existing)
            if enabled and hasattr(self, 'google_vision_compare_checkbox'):
                if self.google_vision_compare_checkbox.isChecked():
                    self.google_vision_compare_checkbox.setChecked(False)

    def show_spell_check_legend(self):
        """Show color legend for spell check"""
        if self.highlighter:
            legend_text = self.highlighter.get_color_legend()
            
            msg_box = QMessageBox(self)
            msg_box.setWindowTitle("Spell Check Color Legend")
            msg_box.setText(legend_text)
            msg_box.setFont(QFont("Courier New", 10))  # Monospace font for better formatting
            msg_box.setStandardButtons(QMessageBox.Ok)
            msg_box.exec_()
        else:
            QMessageBox.information(
                self, 
                "Color Legend", 
                "Spell check is not initialized."
            )

    def show_dictionary_stats(self):
        """Show dictionary statistics"""
        if not self.dict_manager:
            QMessageBox.warning(self, "No Dictionary", "Dictionary manager not initialized.")
            return
        
        stats = self.dict_manager.get_stats()
        
        stats_text = f"""Dictionary Statistics
    ═══════════════════════

    Main Dictionary:        {stats['main']:,} words
    Domain (GBook):         {stats['gbook']:,} words
    Project Words:          {stats['pwords']:,} words
    Correction Pairs:       {stats['cpair']:,} pairs

    Total Words:            {stats['total']:,}
    Format:                 {stats['format']}
    """
        
        msg_box = QMessageBox(self)
        msg_box.setWindowTitle("Dictionary Statistics")
        msg_box.setText(stats_text)
        msg_box.setFont(QFont("Courier New", 10))
        msg_box.setStandardButtons(QMessageBox.Ok)
        msg_box.exec_()

    def add_selected_to_dictionary(self):
        """Add selected word to project dictionary"""
        cursor = self.text_edit.textCursor()
        
        if not cursor.hasSelection():
            QMessageBox.information(
                self, 
                "No Selection", 
                "Please select a word to add to the dictionary."
            )
            return
        
        word = cursor.selectedText().strip()
        
        if not word:
            QMessageBox.warning(self, "Empty Selection", "Selected text is empty.")
            return
        
        # Check if word contains spaces (multiple words)
        if ' ' in word or '\n' in word:
            QMessageBox.warning(
                self, 
                "Invalid Selection", 
                "Please select a single word without spaces."
            )
            return
        
        # Add to dictionary
        if self.dict_manager.add_to_pwords(word):
            # Mark as manually corrected
            if self.spell_checker:
                self.spell_checker.mark_as_corrected(word)
            
            # Rehighlight to show changes
            if self.highlighter:
                self.highlighter.rehighlight()
            
            self.statusBar().showMessage(f"Added '{word}' to project dictionary")
            
            QMessageBox.information(
                self, 
                "Word Added", 
                f"'{word}' has been added to the project dictionary."
            )
        else:
            QMessageBox.warning(
                self, 
                "Already Exists", 
                f"'{word}' is already in the project dictionary."
            )

    def change_keyboard_layout(self, index):
        """Switch keyboard layouts"""
        if index == 0:
            self.statusBar().showMessage("English input mode")
        else:
            self.statusBar().showMessage("Hindi mode - Please enable Hindi input method in your system settings")
            self.text_edit.setFocus()

    def show_hindi_typing_help(self):
        help_text = """<h3>हिंदी में टाइपिंग के लिए मदद</h3>
<p>आप अंग्रेजी अक्षरों का उपयोग करके हिंदी में लिख सकते हैं:</p>
<ul>
    <li><b>namaste</b> लिखने पर <b>नमस्ते</b> बनेगा</li>
    <li><b>main</b> लिखने पर <b>मैं</b> बनेगा</li>
    <li><b>hindi</b> लिखने पर <b>हिंदी</b> बनेगा</li>
</ul>
<p>हिंदी और अंग्रेजी अक्षर: मैपिंग</p>
<ul>
    <li>a = अ | ā = आ</li>
    <li>i = इ | ī = ई</li>
    <li>u = उ | ū = ऊ</li>
    <li>o = ओ | e = ए</li>
    <li>ṛ = ऋ | ai = ऐ</li>
    <li>au = औ | ka = क</li>
    <li>kha = ख | ga = ग</li>
    <li>gha = घ | ṅa = ङ</li>
    <li>ca = च | cha = छ</li>
    <li>ja = ज | jha = झ</li>
    <li>ña = ञ | ṭa = ट</li>
    <li>ṭha = ठ | ḍa = ड</li>
    <li>ḍha = ढ | ṇa = ण</li>
    <li>ta = त | tha = थ</li>
    <li>da = द | dha = ध</li>
    <li>na = न | pha = फ</li>
    <li>ba = ब | bha = भ</li>
    <li>ma = म | ya = य</li>
    <li>ra = र | la = ल</li>
    <li>va = व | śa = श</li>
    <li>ṣa = ष | sa = स</li>
    <li>ha = ह | pa = प</li>
</ul>
<p>आप अपने सिस्टम इनपुट सेटिंग्स से भी हिंदी कीबोर्ड सक्षम कर सकते हैं।</p>"""
        
        msg_box = QMessageBox(self)
        msg_box.setWindowTitle("Hindi Typing Help")
        msg_box.setTextFormat(Qt.RichText)
        msg_box.setText(help_text)
        msg_box.setStandardButtons(QMessageBox.Ok)
        msg_box.exec_()

    # ═══════════════════════════════════════════════════════════════════
    # OCR METHODS
    # ═══════════════════════════════════════════════════════════════════
    
    def open_model_config(self):
        """Open model configuration dialog"""
        dialog = ModelConfigDialog(
            self, 
            current_tessdata_path=self.tessdata_path,
            current_model_name=self.ocr_language
        )
        
        if dialog.exec_() == QDialog.Accepted:
            config = dialog.get_config()
            
            self.tessdata_path = config['tessdata_path']
            self.ocr_language = config['model_name']
            self.ocr_psm = config['psm']
            self.ocr_config = f'--psm {self.ocr_psm}'
            
            # Update UI
            if self.tessdata_path:
                status_msg = f"Model configured: {self.ocr_language} (Custom path: {os.path.basename(self.tessdata_path)})"
            else:
                status_msg = f"Model configured: {self.ocr_language} (System default)"
            
            self.statusBar().showMessage(status_msg)
            
            # Update language combo
            preset_map = {'san': 0, 'hin+eng': 1, 'hin': 2, 'eng': 3}
            if self.ocr_language in preset_map:
                self.ocr_language_combo.setCurrentIndex(preset_map[self.ocr_language])
            
            self.save_ocr_config()
    
    def save_ocr_config(self):
        """Save OCR configuration"""
        try:
            config_data = {
                'tessdata_path': self.tessdata_path,
                'language': self.ocr_language,
                'psm': self.ocr_psm,
                'auto_ocr': self.auto_ocr_enabled
            }
            
            with open("ocr_config.json", 'w') as f:
                json.dump(config_data, f, indent=4)
            
            print("OCR configuration saved")
        except Exception as e:
            print(f"Error saving OCR config: {e}")
    
    def load_ocr_config(self):
        """Load OCR configuration"""
        try:
            config_file = "ocr_config.json"
            if not os.path.exists(config_file):
                return
            
            with open(config_file, 'r') as f:
                config_data = json.load(f)
            
            self.tessdata_path = config_data.get('tessdata_path')
            self.ocr_language = config_data.get('language', 'san')
            self.ocr_psm = config_data.get('psm', 6)
            self.ocr_config = f'--psm {self.ocr_psm}'
            self.auto_ocr_enabled = config_data.get('auto_ocr', False)
            
            print("OCR configuration loaded")
            
            if self.tessdata_path and os.path.exists(self.tessdata_path):
                print(f"Custom tessdata path: {self.tessdata_path}")
                print(f"Language model: {self.ocr_language}")
        except Exception as e:
            print(f"Error loading OCR config: {e}")
    
    def toggle_auto_ocr(self, enabled):
        """Toggle automatic OCR"""
        self.auto_ocr_enabled = enabled
        status = "enabled" if enabled else "disabled"
        self.statusBar().showMessage(f"Auto OCR {status}")

    def change_ocr_language(self, index):
        """Change OCR language"""
        languages = {0: 'san', 1: 'hin+eng', 2: 'hin', 3: 'eng'}
        self.ocr_language = languages.get(index, 'san')
        self.statusBar().showMessage(f"OCR language set to: {self.ocr_language}")

    def run_manual_ocr(self):
        """Manually trigger OCR"""
        if not self.curr_page_pixmap:
            QMessageBox.warning(
                self,
                "No Page Loaded",
                "Please load a PDF page first."
            )
            return
        
        # Check if OCR is already in progress
        if self.current_page in self.ocr_in_progress:
            self.statusBar().showMessage(
                f"OCR already in progress for page {self.current_page + 1}"
            )
            return
        
        # Check if we have cached OCR
        if self.current_page in self.ocr_cache:
            reply = QMessageBox.question(
                self,
                "OCR Already Available",
                f"OCR text is already available for page {self.current_page + 1}.\n\n"
                f"Do you want to re-run OCR?\n"
                f"(This will use current line segments if saved)",
                QMessageBox.Yes | QMessageBox.No,
                QMessageBox.No
            )
            
            if reply == QMessageBox.No:
                return
            
            # Clear cache to force re-run
            del self.ocr_cache[self.current_page]
        
        # Determine which OCR engine to use
        if self.use_google_vision_comparison and self.google_vision_client and self.google_vision_client.enabled:
            self.perform_google_vision_ocr(self.curr_page_pixmap, self.current_page)
        else:
            self.perform_ocr(self.curr_page_pixmap, self.current_page)

    def perform_ocr(self, pixmap, page_num):
        """Perform OCR with caching"""
        if not TESSERACT_AVAILABLE:
            self.statusBar().showMessage("Tesseract not available")
            return
        
        # Check cache
        if page_num in self.ocr_cache:
            self.load_cached_ocr_text(page_num)
            
            # If Google Vision comparison is enabled and we have cached Google Vision result
            if (self.use_google_vision_comparison and 
                page_num in self.google_vision_cache):
                tesseract_text = self.ocr_cache[page_num]
                google_text = self.google_vision_cache[page_num]
                self.apply_comparison_highlighting(tesseract_text, google_text)
            
            self.statusBar().showMessage(f"Loaded cached OCR for page {page_num + 1}")
            return
        
        # Check if in progress
        if page_num in self.ocr_in_progress:
            self.statusBar().showMessage(f"OCR already in progress for page {page_num + 1}")
            return
        
        self.ocr_in_progress.add(page_num)
        self.ocr_progress.setVisible(True)
        self.ocr_progress.setValue(0)
        
        # ═══════════════════════════════════════════════════════════════════
        # CHECK FOR PREDEFINED LINE SEGMENTS
        # ═══════════════════════════════════════════════════════════════════
        predefined_segments = None
        segment_source = "auto-detection"
        
        if page_num in self.line_segments_cache and self.line_segments_cache[page_num]:
            predefined_segments = self.line_segments_cache[page_num]
            segment_source = f"{len(predefined_segments)} saved segments"
            print(f"Using predefined line segments for page {page_num + 1}: {len(predefined_segments)} segments")
        
        # Model info
        model_info_parts = []
        if self.tessdata_path:
            model_info_parts.append(f"Custom model: {os.path.basename(self.tessdata_path)}")
        else:
            model_info_parts.append("System default")
        
        if foundir_enabled := (self.foundir_enabled and self.foundir_client.enabled):
            model_info_parts.append("FoundIR denoising")
        
        if self.use_google_vision_comparison:
            model_info_parts.append("+ Google Vision comparison")
        
        model_info_parts.append(segment_source)
        model_info = f"({', '.join(model_info_parts)})"
        
        self.statusBar().showMessage(
            f"Running OCR on page {page_num + 1} with {self.ocr_language} {model_info}..."
        )
        
        # Create worker with predefined segments
        worker = OCRWorker(
            pixmap, 
            page_num, 
            self.ocr_language, 
            self.ocr_config,
            self.tessdata_path,
            self.spell_checker,
            self,
            self.google_vision_client,
            predefined_segments  # Pass predefined segments
        )
        worker.result_ready.connect(self.handle_ocr_result)
        worker.google_vision_result_ready.connect(self.handle_google_vision_result)
        worker.progress_update.connect(self.update_ocr_progress)
        worker.error_occurred.connect(self.handle_ocr_error)
        self.ocr_workers.append(worker)
        worker.start()
    
    def handle_ocr_result(self, text, page_num, spell_check_results=None, line_segments=None):
        """Handle OCR result and cache it"""
        # Store in cache
        self.ocr_cache[page_num] = text

        # Store line segments
        if line_segments:
            self.line_segments_cache[page_num] = line_segments

            # Enable controls
            self.show_line_segments_checkbox.setEnabled(True)
            self.edit_line_segments_checkbox.setEnabled(True)
            self.add_line_segment_checkbox.setEnabled(True)
            self.load_line_segments_button.setEnabled(True)    
            
        # Remove from in-progress set
        self.ocr_in_progress.discard(page_num)
        
        # Only update text editor if this is the current page
        if page_num == self.current_page:
            # Check if line segments should be displayed
            if self.show_line_segments_checkbox.isChecked() and line_segments:
                self.display_line_segments(line_segments)
            # ═══════════════════════════════════════════════════════════════
            # DECISION: Which highlighting mode to use?
            # ═══════════════════════════════════════════════════════════════
            
            # Check if Google Vision comparison is enabled and we have Google Vision result
            if (self.use_google_vision_comparison and 
                page_num in self.google_vision_cache):
                # Mode 1: Google Vision Comparison (spell check disabled)
                google_text = self.google_vision_cache[page_num]
                self.apply_comparison_highlighting(text, google_text)
                self.statusBar().showMessage(
                    f"OCR completed for page {page_num + 1} with Google Vision comparison ({len(text)} characters)"
                )
            
            elif self.spell_check_enabled_checkbox.isChecked() and self.highlighter.enabled:
                # Mode 2: Spell Check (Google Vision disabled)
                self.load_cached_ocr_text(page_num)
                self.statusBar().showMessage(
                    f"OCR completed for page {page_num + 1} with spell check ({len(text)} characters)"
                )
            
            else:
                # Mode 3: Plain text (both disabled)
                self.load_cached_ocr_text(page_num)
                self.statusBar().showMessage(
                    f"OCR completed for page {page_num + 1} ({len(text)} characters)"
                )
        else:
            self.statusBar().showMessage(f"OCR cached for page {page_num + 1}")
    
        self.ocr_progress.setVisible(False)
        
        # Clean up finished workers
        self.ocr_workers = [w for w in self.ocr_workers if w.isRunning()]
    
    # ═══════════════════════════════════════════════════════════════════
    # IMAGE COMPARISON AND SIMILARITY
    # ═══════════════════════════════════════════════════════════════════
    
    def build_hash_index(self):
        """Build hash index for fast similarity search"""
        if not IMAGEHASH_AVAILABLE:
            return
        
        try:
            hashes = []
            image_ids = []
            
            for image_id, data in self.image_lookup_data.items():
                image_path = data.get('path')
                if not image_path or not os.path.exists(image_path):
                    continue
                
                try:
                    img = Image.open(image_path)
                    hash_val = imagehash.dhash(img, hash_size=8)
                    hashes.append(list(hash_val.hash.flatten()))
                    image_ids.append(image_id)
                except Exception as e:
                    print(f"Error hashing image {image_id}: {e}")
                    continue
            
            if hashes:
                self.hash_tree = KDTree(hashes)
                self.hash_ids = image_ids
                print(f"Built hash index with {len(hashes)} images")
            
        except Exception as e:
            print(f"Error building hash index: {e}")
            self.hash_tree = None

    @lru_cache(maxsize=1024)
    def compare_images_cached(self, img1_data, img2_data):
        """Cached image comparison using MSE"""
        img1_array = np.frombuffer(img1_data, dtype=np.uint8).reshape(FIXED_IMAGE_SIZE[1], FIXED_IMAGE_SIZE[0], 3)
        img2_array = np.frombuffer(img2_data, dtype=np.uint8).reshape(FIXED_IMAGE_SIZE[1], FIXED_IMAGE_SIZE[0], 3)
        
        err = np.sum((img1_array.astype("float") - img2_array.astype("float")) ** 2)
        err /= float(img1_array.size)
        
        return 1.0 / (1.0 + err) if err > 0 else 1.0

    def compare_images(self, img1, img2):
        """Compare two images using MSE"""
        try:
            # Resize to fixed size
            img1_resized = img1.scaled(*FIXED_IMAGE_SIZE, Qt.IgnoreAspectRatio, 
                                       Qt.SmoothTransformation).toImage()
            img2_resized = img2.scaled(*FIXED_IMAGE_SIZE, Qt.IgnoreAspectRatio, 
                                       Qt.SmoothTransformation).toImage()
            
            # Convert to numpy and cache
            img1_array = qimage_to_numpy(img1_resized)
            img2_array = qimage_to_numpy(img2_resized)
            
            # Use cached comparison
            img1_bytes = img1_array.tobytes()
            img2_bytes = img2_array.tobytes()
            
            return self.compare_images_cached(img1_bytes, img2_bytes)
            
        except Exception as e:
            print(f"Error in image comparison: {e}")
            return 0.01

    def get_similar_images(self, selected_image, num_similar=10):
        """Find similar images"""
        if not self.image_lookup_data:
            print("No images in database")
            return []
        
        # Use hash-based search if available
        if IMAGEHASH_AVAILABLE and self.hash_tree is not None:
            try:
                return self.get_similar_images_fast(selected_image, num_similar)
            except Exception as e:
                print(f"Fast search failed, falling back: {e}")
                return self.get_similar_images_slow(selected_image, num_similar)
        else:
            return self.get_similar_images_slow(selected_image, num_similar)

    def get_similar_images_fast(self, selected_image, num_similar=10):
        """Fast similarity search using perceptual hashing"""
        img_pil = qimage_to_pil(selected_image)
        query_hash = imagehash.dhash(img_pil, hash_size=8)
        query_vector = list(query_hash.hash.flatten())
        
        if not self.hash_tree or not self.hash_ids:
            print("Hash tree not available")
            return self.get_similar_images_slow(selected_image, num_similar)
        
        k = min(num_similar + 1, len(self.hash_ids))
        
        if k <= 0:
            return []
        
        # Query KD-tree
        result = self.hash_tree.query(query_vector, k=k)
        
        # Handle different return formats
        if k == 1:
            distances = [float(result[0])]
            indices = [int(result[1])]
        else:
            distances, indices = result
            distances = np.atleast_1d(distances)
            indices = np.atleast_1d(indices)
        
        # Build results
        similar = []
        for dist, idx in zip(distances, indices):
            idx = int(idx)
            
            if idx < 0 or idx >= len(self.hash_ids):
                continue
                
            image_id = self.hash_ids[idx]
            image_data = self.image_lookup_data.get(image_id)
            
            if not image_data:
                continue
            
            image_path = image_data.get('path')
            if not image_path or not os.path.exists(image_path):
                continue
            
            try:
                pixmap = QPixmap(image_path)
                if pixmap.isNull():
                    continue
                    
                similarity = 1.0 / (1.0 + float(dist))
                similar.append((similarity, pixmap, image_id))
            except Exception as e:
                print(f"Error loading image {image_id}: {e}")
                continue
        
        return similar[:num_similar]

    def get_similar_images_slow(self, selected_image, num_similar=10):
        """Fallback similarity search using MSE"""
        similarities = []
        selected_pixmap = QPixmap.fromImage(selected_image)
        
        for image_id, image_data in self.image_lookup_data.items():
            image_path = image_data.get('path')
            
            if not image_path or not os.path.exists(image_path):
                continue
            
            try:
                comparison_pixmap = QPixmap(image_path)
                similarity = self.compare_images(selected_pixmap, comparison_pixmap)
                similarities.append((similarity, comparison_pixmap, image_id))
            except Exception as e:
                print(f"Error comparing image {image_id}: {e}")
                continue
        
        # Sort by similarity
        similarities.sort(reverse=True, key=lambda x: x[0])
        return similarities[:num_similar]

    def save_selections_to_images(self):
        """Save all user selections as images"""
        if not hasattr(self.pdf_content, "user_selections") or not self.pdf_content.user_selections:
            QMessageBox.warning(self, "No Selections", "No akshara selections to save.")
            return
        
        if not self.curr_page_pixmap:
            return
        
        try:
            image = self.curr_page_pixmap.toImage()
            base_filename = os.path.splitext(self.current_pdf_filename)[0] or f"image_page_{self.current_page + 1}"
            
            # Extract selected images
            selected_pixmaps = []
            selected_data = []
            
            for i, (box, original_box) in enumerate(zip(self.pdf_content.user_selections, 
                                                        self.pdf_content.user_selections_original)):
                x, y = original_box.x(), original_box.y()
                width, height = original_box.width(), original_box.height()
                
                cropped = image.copy(x, y, width, height)
                selected_pixmaps.append(QPixmap.fromImage(cropped))
                selected_data.append({
                    'box': box,
                    'original_box': original_box,
                    'x': x, 'y': y,
                    'width': width, 'height': height,
                    'index': i
                })
            
            # Find similar images
            if selected_pixmaps:
                first_image = selected_pixmaps[0].toImage()
                
                similar_images = []
                try:
                    similar_images = self.get_similar_images(first_image, num_similar=10)
                    print(f"Found {len(similar_images)} similar images")
                except Exception as e:
                    print(f"Error finding similar images: {e}")
                    similar_images = []
                
                # Show comparison dialog
                dialog = ImageComparisonDialog(self, selected_pixmaps, similar_images)
                
                if dialog.exec_() == QDialog.Accepted:
                    results = dialog.get_results()
                    selected_image_ids = results['selected_images']
                    is_insert_underscore = results['insert_underscore']
                    character = results['character']
                    
                    # Insert for newly selected aksharas
                    num_aksharas = len(self.pdf_content.user_selections)
                    if is_insert_underscore:
                        self.insert_underscores_at_cursor(num_aksharas)
                    else:
                        self.insert_characters_at_cursor(character, num_aksharas)
                    
                    # Handle similar images
                    if selected_image_ids:
                        if is_insert_underscore:
                            self.insert_underscores_at_cursor(len(selected_image_ids))
                        else:
                            self.insert_and_replace_similar_images(selected_image_ids, character)
                
                # Save all images
                for data in selected_data:
                    cropped = image.copy(data['x'], data['y'], data['width'], data['height'])
                    self.save_selected_image(
                        QPixmap.fromImage(cropped),
                        data['original_box'],
                        f"{base_filename}_{data['index']}"
                    )
                
                # Rebuild hash index
                if IMAGEHASH_AVAILABLE:
                    self.build_hash_index()
                
                # Clear selections
                self.pdf_content.clear_user_selections()
                self.pdf_content.update()
                self.save_selections_button.setEnabled(False)
                
                self.statusBar().showMessage(f"Saved {len(selected_data)} akshara selections")
            
        except Exception as e:
            QMessageBox.critical(
                self, 
                "Error Processing Selections", 
                f"An error occurred:\n{str(e)}"
            )

    def save_selected_image(self, cropped_image, original_box, base_filename):
        """Save selected image to lookup directory"""
        try:
            image_filename = f"{base_filename}_char_{self.next_image_id}.png"
            image_path = os.path.join(self.image_lookup_dir, image_filename)
            
            if isinstance(cropped_image, QPixmap):
                cropped_image = cropped_image.toImage()
            
            # Save with atomic operation
            save_image_safely(cropped_image, image_path)
            
            # Add to lookup
            self.image_lookup_data[str(self.next_image_id)] = {
                'path': image_path,
                'id': self.next_image_id,
                'char': '_',
                'page': self.current_page + 1,
                'bbox': [original_box.x(), original_box.y(), 
                        original_box.width(), original_box.height()],
                'source_file': self.current_pdf_filename,
                'inserted': False,
                'text_position': -1
            }
            
            self.next_image_id += 1
            self.save_image_lookup_data()
            
        except Exception as e:
            print(f"Error saving image: {e}")
            raise

    def insert_underscores_at_cursor(self, count):
        """Insert underscores at cursor"""
        if count <= 0:
            return
        
        cursor = self.text_edit.textCursor()
        
        for i in range(count):
            cursor.insertText("_")
        
        self.text_edit.setTextCursor(cursor)
        self.statusBar().showMessage(f"Inserted {count} underscore(s)")
    
    def insert_characters_at_cursor(self, character, count):
        """Insert characters at cursor"""
        if count <= 0:
            return
        
        cursor = self.text_edit.textCursor()
    
        for i in range(count):
            cursor.insertText(character)
        
        self.text_edit.setTextCursor(cursor)
        self.statusBar().showMessage(f"Inserted {count} '{character}' character(s)")

    def insert_and_replace_similar_images(self, image_ids, character):
        """Insert character and replace occurrences"""
        if not image_ids or not character:
            return
        
        current_text = self.text_edit.toPlainText()
        cursor = self.text_edit.textCursor()
        cursor_position = cursor.position()
        
        # Collect information
        similar_image_info = []
        for image_id in image_ids:
            if image_id in self.image_lookup_data:
                img_data = self.image_lookup_data[image_id]
                old_char = img_data.get('char', '_')
                page_num = img_data.get('page', 'Unknown')
                
                similar_image_info.append({
                    'id': image_id,
                    'old_char': old_char,
                    'page': page_num
                })
                
                self.image_lookup_data[image_id]['char'] = character
                print(f"Image {image_id} (Page {page_num}): '{old_char}' → '{character}'")
        
        self.save_image_lookup_data()
        
        # Group by page
        pages_dict = {}
        for info in similar_image_info:
            page = info['page']
            pages_dict[page] = pages_dict.get(page, 0) + 1
        
        page_info = ", ".join([f"Page {pg}: {cnt}" for pg, cnt in sorted(pages_dict.items())])
        
        # Find and replace underscores
        num_underscores = current_text.count('_')
        
        if num_underscores > 0:
            message = (
                f'Found {num_underscores} underscore(s) in text.\n'
                f'Replace {len(image_ids)} underscore(s) with "{character}"?\n\n'
                f'Similar aksharas found on:\n{page_info}\n\n'
                f'This will replace the next {len(image_ids)} underscores from cursor position.'
            )
            
            reply = QMessageBox.question(
                self,
                'Replace Underscores',
                message,
                QMessageBox.Yes | QMessageBox.No,
                QMessageBox.Yes
            )
            
            if reply == QMessageBox.Yes:
                replacements_made = 0
                text_before_cursor = current_text[:cursor_position]
                text_after_cursor = current_text[cursor_position:]
                
                new_text_after = ""
                underscore_count = 0
                position_in_text = cursor_position
                
                for char in text_after_cursor:
                    if char == '_' and underscore_count < len(image_ids):
                        new_text_after += character
                        underscore_count += 1
                        replacements_made += 1
                        
                        if underscore_count <= len(similar_image_info):
                            img_id = similar_image_info[underscore_count - 1]['id']
                            self.image_lookup_data[img_id]['inserted'] = True
                            self.image_lookup_data[img_id]['text_position'] = position_in_text
                        
                        position_in_text += 1
                    else:
                        new_text_after += char
                        position_in_text += 1
                
                new_text = text_before_cursor + new_text_after
                
                self.text_edit.setPlainText(new_text)
                
                cursor = self.text_edit.textCursor()
                cursor.setPosition(cursor_position)
                self.text_edit.setTextCursor(cursor)
                
                self.save_image_lookup_data()
                
                self.statusBar().showMessage(
                    f"Replaced {replacements_made} underscore(s) with '{character}' "
                    f"(from {len(pages_dict)} page(s))"
                )
                return
        
        # No replacement, just insert
        cursor = self.text_edit.textCursor()
        insert_position = cursor.position()
        
        for i, info in enumerate(similar_image_info):
            cursor.insertText(character)
            
            img_id = info['id']
            self.image_lookup_data[img_id]['inserted'] = True
            self.image_lookup_data[img_id]['text_position'] = insert_position + i
        
        self.text_edit.setTextCursor(cursor)
        self.save_image_lookup_data()
        
        self.statusBar().showMessage(
            f"Inserted {len(image_ids)} '{character}' at cursor position "
            f"(from {len(pages_dict)} page(s))"
        )

    def load_image_lookup_data(self):
        """Load image lookup data"""
        if not os.path.exists(self.image_lookup_file):
            self.image_lookup_data = {}
            self.next_image_id = 1
            return
        
        try:
            with open(self.image_lookup_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                if not isinstance(data, dict):
                    raise ValueError("Invalid JSON structure")
                
                images = data.get('images', {})
                if not isinstance(images, dict):
                    raise ValueError("Invalid images structure")
                
                self.image_lookup_data = images
                self.next_image_id = data.get('next_id', 1)
                
                if not isinstance(self.next_image_id, int) or self.next_image_id < 1:
                    self.next_image_id = 1
                
                print(f"Loaded {len(self.image_lookup_data)} images")
                
        except (json.JSONDecodeError, ValueError) as e:
            print(f"Error loading data: {e}")
            QMessageBox.warning(
                self,
                "Data Load Error",
                "Could not load image lookup data. Starting fresh."
            )
            self.image_lookup_data = {}
            self.next_image_id = 1

    def save_image_lookup_data(self):
        """Save image lookup data"""
        try:
            data = {
                'images': self.image_lookup_data,
                'next_id': self.next_image_id
            }
            
            # Atomic write
            temp_file = self.image_lookup_file + '.tmp'
            with open(temp_file, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            
            if not os.path.exists(temp_file):
                raise IOError("Temporary file not created")
            
            os.replace(temp_file, self.image_lookup_file)
            
        except Exception as e:
            print(f"Error saving data: {e}")
            QMessageBox.warning(
                self,
                "Save Error",
                f"Could not save image lookup data:\n{str(e)}"
            )

    # ═══════════════════════════════════════════════════════════════════
    # Unified Document Loading
    # ═══════════════════════════════════════════════════════════════════
    
    def load_document(self):
        """
        Load either a PDF or an image file
        Unified entry point for both document types
        """
        file_path, _ = QFileDialog.getOpenFileName(
            self, 
            "Open PDF or Image File", 
            "", 
            "All Supported (*.pdf *.png *.jpg *.jpeg *.bmp *.tiff *.tif);;PDF Files (*.pdf);;Image Files (*.png *.jpg *.jpeg *.bmp *.tiff *.tif);;All Files (*.*)"
        )
        
        if not file_path:
            return
        
        # Determine file type by extension
        file_ext = os.path.splitext(file_path)[1].lower()
        
        if file_ext == '.pdf':
            self.load_pdf_document(file_path)
        elif file_ext in ['.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif']:
            self.load_image_document(file_path)
        else:
            QMessageBox.warning(
                self,
                "Unsupported Format",
                f"File format '{file_ext}' is not supported.\n"
                "Supported formats: PDF, PNG, JPG, JPEG, BMP, TIFF"
            )

    def load_pdf_document(self, file_path):
        """Load PDF document (existing functionality)"""
        try:
            if self.pdf_document:
                self.pdf_document.close()

            # Clear caches
            self.ocr_cache.clear()
            self.ocr_in_progress.clear()
            self.page_selections.clear()
            
            self.pdf_document = fitz.open(file_path)
            self.total_pages = len(self.pdf_document)
            self.current_page = 0
            self.current_pdf_filename = os.path.basename(file_path)
            
            # Set document type
            self.current_document_type = 'pdf'
            self.current_image_path = None
            
            # Update UI
            self.page_label.setText(f"Page {self.current_page + 1} of {self.total_pages}")
            self.prev_button.setEnabled(False)
            self.next_button.setEnabled(self.total_pages > 1)
            
            # Enable controls
            self.enable_document_controls()
            
            # Reset zoom
            self.zoom_reset()
            self.clear_selection()
            self.load_page(0)
            
            self.statusBar().showMessage(
                f"Loaded PDF: {self.current_pdf_filename} ({self.total_pages} pages)"
            )
        
        except Exception as e:
            QMessageBox.critical(
                self,
                "Error Loading PDF",
                f"Could not load PDF:\n{str(e)}"
            )
            self.statusBar().showMessage(f"Error loading PDF: {str(e)}")

    def load_image_document(self, file_path):
        """
        Load image file as a single-page document
        NEW functionality for image OCR
        """
        try:
            # Load image as pixmap
            pixmap = load_image_as_pixmap(file_path)
            
            if pixmap is None or pixmap.isNull():
                raise Exception("Failed to load image file")
            
            # Close any open PDF
            if self.pdf_document:
                self.pdf_document.close()
                self.pdf_document = None
            
            # Clear caches
            self.ocr_cache.clear()
            self.ocr_in_progress.clear()
            self.page_selections.clear()
            
            # Set document properties
            self.total_pages = 1
            self.current_page = 0
            self.current_pdf_filename = os.path.basename(file_path)
            self.curr_page_pixmap = pixmap
            
            # Set document type
            self.current_document_type = 'image'
            self.current_image_path = file_path
            
            # Update UI
            self.page_label.setText(f"Image 1 of 1")
            self.prev_button.setEnabled(False)
            self.next_button.setEnabled(False)  # Single image, no navigation
            
            # Enable controls
            self.enable_document_controls()
            
            # Reset zoom and display
            self.zoom_reset()
            self.clear_selection()
            self.apply_zoom()
            
            # Clear text editor
            self.text_edit.clear()
            
            self.statusBar().showMessage(
                f"Loaded Image: {self.current_pdf_filename} ({pixmap.width()}x{pixmap.height()})"
            )
        
        except Exception as e:
            QMessageBox.critical(
                self,
                "Error Loading Image",
                f"Could not load image:\n{str(e)}"
            )
            self.statusBar().showMessage(f"Error loading image: {str(e)}")

    def enable_document_controls(self):
        """Enable controls after loading a document"""
        self.zoom_out_button.setEnabled(True)
        self.zoom_in_button.setEnabled(True)
        self.zoom_slider.setEnabled(True)
        self.zoom_reset_button.setEnabled(True)
        self.selection_checkbox.setEnabled(True)
        self.save_text_button.setEnabled(True)
        self.load_text_button.setEnabled(True)
        self.manual_ocr_button.setEnabled(TESSERACT_AVAILABLE)
        self.load_line_segments_button.setEnabled(True) 
        self.add_line_segment_checkbox.setEnabled(True)
    
    # ═══════════════════════════════════════════════════════════════════
    # PDF LOADING AND NAVIGATION
    # ═══════════════════════════════════════════════════════════════════

    def load_pdf(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open PDF File", "", "PDF Files (*.pdf)")
        
        if not file_path:
            return
        
        try:
            if self.pdf_document:
                self.pdf_document.close()

            # Clear caches
            self.ocr_cache.clear()
            self.ocr_in_progress.clear()
            self.page_selections.clear()
            
            self.pdf_document = fitz.open(file_path)
            self.total_pages = len(self.pdf_document)
            self.current_page = 0
            self.current_pdf_filename = os.path.basename(file_path)
            
            # Update UI
            self.page_label.setText(f"Page {self.current_page + 1} of {self.total_pages}")
            self.prev_button.setEnabled(False)
            self.next_button.setEnabled(self.total_pages > 1)
            
            # Enable controls
            self.zoom_out_button.setEnabled(True)
            self.zoom_in_button.setEnabled(True)
            self.zoom_slider.setEnabled(True)
            self.zoom_reset_button.setEnabled(True)
            self.selection_checkbox.setEnabled(True)
            self.save_text_button.setEnabled(True)
            self.load_text_button.setEnabled(True)
            self.manual_ocr_button.setEnabled(TESSERACT_AVAILABLE)
            self.load_line_segments_button.setEnabled(True) 
            self.add_line_segment_checkbox.setEnabled(True)
            
            # Reset zoom
            self.zoom_reset()
            self.clear_selection()
            self.load_page(0)
            
            self.statusBar().showMessage(
                f"Loaded: {self.current_pdf_filename} ({self.total_pages} pages)"
            )
        
        except Exception as e:
            QMessageBox.critical(
                self,
                "Error Loading PDF",
                f"Could not load PDF:\n{str(e)}"
            )
            self.statusBar().showMessage(f"Error loading PDF: {str(e)}")

    def load_page(self, page_num):
        """
        Load page - works for both PDF and image documents
        """
        if self.current_document_type == 'pdf':
            self.load_pdf_page(page_num)
        elif self.current_document_type == 'image':
            self.load_image_page()
        else:
            return
        
    def load_pdf_page(self, page_num):
        """Load PDF page (existing functionality)"""
        if not self.pdf_document or page_num < 0 or page_num >= self.total_pages:
            return
        
        try:
            # Save current page selections before changing page
            if hasattr(self.pdf_content, 'user_selections') and self.pdf_content.user_selections:
                self.page_selections[self.current_page] = list(zip(
                    self.pdf_content.user_selections,
                    self.pdf_content.user_selections_original
                ))
            
            # Reset current selection
            self.current_selection = None
            self.current_selection_original = None
            
            # Load page from PDF
            page = self.pdf_document.load_page(page_num)
            
            # Render page to pixmap with base zoom
            zoom_matrix = fitz.Matrix(BASE_RENDER_ZOOM, BASE_RENDER_ZOOM)
            pix = page.get_pixmap(matrix=zoom_matrix)
            
            # Convert to QPixmap
            img_data = pix.tobytes("ppm")
            qimg = QImage.fromData(img_data)
            pixmap = QPixmap.fromImage(qimg)
            
            # Store original pixmap
            self.curr_page_pixmap = pixmap
            
            # Update page info
            self.current_page = page_num
            
            # Apply zoom
            self.apply_zoom()
            
            # Restore selections
            if page_num in self.page_selections:
                for rect, original_rect in self.page_selections[page_num]:
                    scaled_rect = QRect(
                        int(original_rect.x() * self.zoom_factor),
                        int(original_rect.y() * self.zoom_factor),
                        int(original_rect.width() * self.zoom_factor),
                        int(original_rect.height() * self.zoom_factor)
                    )
                    self.pdf_content.add_user_selection(scaled_rect, original_rect)
                
                self.save_selections_button.setEnabled(True)
            
            self.page_label.setText(f"Page {page_num + 1} of {self.total_pages}")
            
            # Update navigation buttons
            self.prev_button.setEnabled(page_num > 0)
            self.next_button.setEnabled(page_num < self.total_pages - 1)
            
            # Load cached OCR if available
            self.load_cached_ocr_if_available(page_num)
            
            # Display line segments if needed
            if (self.show_line_segments_checkbox.isChecked() and 
                page_num in self.line_segments_cache):
                self.display_line_segments(self.line_segments_cache[page_num])
                        
        except Exception as e:
            QMessageBox.critical(
                self,
                "Error Loading Page",
                f"Could not load page {page_num + 1}:\n{str(e)}"
            )

    def load_image_page(self):
        """
        Load image as single page
        NEW: Handles image display
        """
        try:
            # For images, we already have the pixmap loaded
            if not self.curr_page_pixmap:
                return
            
            # Reset selection
            self.current_selection = None
            self.current_selection_original = None
            
            # Apply zoom
            self.apply_zoom()
            
            # Restore selections if any
            if 0 in self.page_selections:
                for rect, original_rect in self.page_selections[0]:
                    scaled_rect = QRect(
                        int(original_rect.x() * self.zoom_factor),
                        int(original_rect.y() * self.zoom_factor),
                        int(original_rect.width() * self.zoom_factor),
                        int(original_rect.height() * self.zoom_factor)
                    )
                    self.pdf_content.add_user_selection(scaled_rect, original_rect)
                
                self.save_selections_button.setEnabled(True)
            
            # Load cached OCR if available
            self.load_cached_ocr_if_available(0)
            
            # Display line segments if needed
            if (self.show_line_segments_checkbox.isChecked() and 
                0 in self.line_segments_cache):
                self.display_line_segments(self.line_segments_cache[0])
                        
        except Exception as e:
            QMessageBox.critical(
                self,
                "Error Displaying Image",
                f"Could not display image:\n{str(e)}"
            )

    def load_cached_ocr_if_available(self, page_num):
        """Helper method to load cached OCR text"""
        if page_num in self.ocr_cache:
            tesseract_text = self.ocr_cache[page_num]
            
            if (self.use_google_vision_comparison and 
                page_num in self.google_vision_cache):
                google_text = self.google_vision_cache[page_num]
                self.apply_comparison_highlighting(tesseract_text, google_text)
                self.statusBar().showMessage(
                    f"Page {page_num + 1} loaded (Google Vision comparison)"
                )
            
            elif self.spell_check_enabled_checkbox.isChecked():
                self.load_cached_ocr_text(page_num)
                self.statusBar().showMessage(
                    f"Page {page_num + 1} loaded (spell check enabled)"
                )
            
            else:
                self.load_cached_ocr_text(page_num)
                doc_type = "Image" if self.current_document_type == 'image' else f"Page {page_num + 1}"
                self.statusBar().showMessage(f"{doc_type} loaded (cached OCR)")
        
        else:
            self.text_edit.clear()
            doc_type = "Image" if self.current_document_type == 'image' else f"Page {page_num + 1}"
            self.statusBar().showMessage(
                f"{doc_type} loaded - Click 'Run OCR Now' to extract text"
            )

    def next_page(self):
        if self.current_document_type != 'pdf':
            return
        if self.current_page < self.total_pages - 1:
            self.load_page(self.current_page + 1)

    def prev_page(self):
        if self.current_document_type != 'pdf':
            return
        if self.current_page > 0:
            self.load_page(self.current_page - 1)

    # ═══════════════════════════════════════════════════════════════════
    # ZOOM METHODS
    # ═══════════════════════════════════════════════════════════════════

    def wheelEvent(self, event):
        """Handle mouse wheel for zooming"""
        if event.modifiers() & Qt.ControlModifier:
            delta = event.angleDelta().y()
            if delta > 0:
                self.zoom_in()
            else:
                self.zoom_out()
            event.accept()
        else:
            super().wheelEvent(event)

    def zoom_in(self):
        if not self.zoom_slider.isEnabled():
            return
        new_value = min(self.zoom_slider.value() + ZOOM_STEP, self.zoom_slider.maximum())
        self.zoom_slider.setValue(new_value)
        
    def zoom_out(self):
        if not self.zoom_slider.isEnabled():
            return
        new_value = max(self.zoom_slider.value() - ZOOM_STEP, self.zoom_slider.minimum())
        self.zoom_slider.setValue(new_value)
        
    def zoom_reset(self):
        if not self.zoom_slider.isEnabled():
            return
        self.zoom_slider.setValue(DEFAULT_ZOOM)

    def zoom_slider_changed(self, value):
        self.zoom_factor = value / 100.0
        self.zoom_label.setText(f"{value}%")
        self.apply_zoom()

    def apply_zoom(self):
        """Apply zoom factor"""
        if not self.curr_page_pixmap:
            return
        
        try:
            # Scale pixmap
            scaled_pixmap = self.curr_page_pixmap.scaled(
                int(self.curr_page_pixmap.width() * self.zoom_factor),
                int(self.curr_page_pixmap.height() * self.zoom_factor),
                Qt.KeepAspectRatio,
                Qt.SmoothTransformation
            )
            
            # Get current states
            selection_mode = getattr(self.pdf_content, "selection_mode", False)
            user_selections_original = getattr(self.pdf_content, "user_selections_original", [])
            show_line_segments = self.show_line_segments_checkbox.isChecked()
            editing_line_segments = self.edit_line_segments_checkbox.isChecked()
            adding_line_segments = self.add_line_segment_checkbox.isChecked()
            
            # Create new view
            self.pdf_content = PDFViewLabel()
            self.pdf_content.setPixmap(scaled_pixmap)
            self.pdf_content.setAlignment(Qt.AlignCenter)
            self.pdf_content.selectionChanged.connect(self.update_selection)
            self.pdf_content.selectionFinished.connect(self.finalize_selection)
            self.pdf_content.enable_selection(selection_mode)
            
            # Restore selections
            if user_selections_original:
                for original_box in user_selections_original:
                    scaled_box = QRect(
                        int(original_box.x() * self.zoom_factor),
                        int(original_box.y() * self.zoom_factor),
                        int(original_box.width() * self.zoom_factor),
                        int(original_box.height() * self.zoom_factor)
                    )
                    self.pdf_content.add_user_selection(scaled_box, original_box)
            
            # Restore line segments if they should be shown
            if show_line_segments and self.current_page in self.line_segments_cache:
                self.display_line_segments(self.line_segments_cache[self.current_page])

            # Restore line segment editing mode
            if editing_line_segments:
                self.pdf_content.enable_line_segment_editing(True)
                self.pdf_content.lineSegmentChanged.connect(self.handle_line_segment_changed)

            # Restore add line segment mode
            if adding_line_segments:
                self.pdf_content.enable_add_line_segment(True)
                self.pdf_content.newLineSegmentCreated.connect(self.handle_new_line_segment_created)
                
            # Restore current selection
            if self.current_selection and not self.current_selection.isEmpty():
                scaled_selection = QRect(
                    int(self.current_selection_original.x() * self.zoom_factor),
                    int(self.current_selection_original.y() * self.zoom_factor),
                    int(self.current_selection_original.width() * self.zoom_factor),
                    int(self.current_selection_original.height() * self.zoom_factor)
                )
                self.pdf_content.current_rect = scaled_selection
            
            self.pdf_scroll_area.setWidget(self.pdf_content)
            
        except Exception as e:
            print(f"Error applying zoom: {e}")

    # ═══════════════════════════════════════════════════════════════════
    # SELECTION METHODS
    # ═══════════════════════════════════════════════════════════════════

    def toggle_selection_mode(self, enabled):
        if hasattr(self.pdf_content, "enable_selection"):
            self.pdf_content.enable_selection(enabled)
        
        self.clear_selection_button.setEnabled(enabled)
        self.add_selection_button.setEnabled(enabled)
        self.save_selections_button.setEnabled(
            enabled and bool(getattr(self.pdf_content, "user_selections", []))
        )
        
        if not enabled:
            self.current_selection = None
            self.current_selection_original = None
        
        status = "enabled" if enabled else "disabled"
        self.statusBar().showMessage(f"Selection mode {status}")

    def update_selection(self, rect):
        if not rect.isEmpty():
            self.statusBar().showMessage(
                f"Selection: ({rect.x()}, {rect.y()}, {rect.width()}x{rect.height()})"
            )

    def finalize_selection(self, rect):
        if rect.isEmpty():
            self.statusBar().showMessage("Selection cleared")
            self.add_selection_button.setEnabled(False)
            self.current_selection = None
            self.current_selection_original = None
            return
        
        # Store coordinates
        self.current_selection = rect
        self.current_selection_original = QRect(
            int(rect.x() / self.zoom_factor),
            int(rect.y() / self.zoom_factor),
            int(rect.width() / self.zoom_factor),
            int(rect.height() / self.zoom_factor)
        )
        
        self.statusBar().showMessage(
            f"Selection finalized: ({rect.x()}, {rect.y()}, {rect.width()}x{rect.height()})"
        )
        self.add_selection_button.setEnabled(True)

    def add_selection(self):
        """Add current selection"""
        if self.current_selection and not self.current_selection.isEmpty():
            if self.pdf_content.add_user_selection(
                self.current_selection, 
                self.current_selection_original
            ):
                num_selections = len(self.pdf_content.user_selections)
                self.statusBar().showMessage(f"Added selection ({num_selections} total)")
                self.save_selections_button.setEnabled(True)
                
                # Clear current
                self.pdf_content.current_rect = QRect()
                self.current_selection = None
                self.current_selection_original = None
                self.pdf_content.update()

    def clear_all_selections_on_page(self):
        """Clear all page selections"""
        if hasattr(self.pdf_content, "clear_user_selections"):
            self.pdf_content.clear_user_selections()
            self.pdf_content.update()
        
        if self.current_page in self.page_selections:
            del self.page_selections[self.current_page]
        
        self.current_selection = None
        self.current_selection_original = None
        self.save_selections_button.setEnabled(False)
        
        self.statusBar().showMessage("All selections cleared")

    def clear_selection(self):
        """Clear current selection"""
        if hasattr(self.pdf_content, "current_rect"):
            self.pdf_content.current_rect = QRect()
            self.pdf_content.update()
        
        self.current_selection = None
        self.current_selection_original = None
        
        has_selections = (hasattr(self.pdf_content, 'user_selections') and 
                        bool(self.pdf_content.user_selections))
        self.save_selections_button.setEnabled(has_selections)
        
        self.statusBar().showMessage("Current selection cleared")

    # ═══════════════════════════════════════════════════════════════════
    # TEXT FILE OPERATIONS
    # ═══════════════════════════════════════════════════════════════════

    def load_text_to_editor(self):
        """Load text from file"""
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Load Text File", "", "Text Files (*.txt);;All Files (*.*)"
        )
        
        if not file_path:
            return
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
            
            cursor = self.text_edit.textCursor()
            cursor.insertText(text)
            
            self.statusBar().showMessage(f"Text loaded from {os.path.basename(file_path)}")
            
        except Exception as e:
            QMessageBox.critical(
                self,
                "Error Loading File",
                f"An error occurred:\n{str(e)}"
            )

    def save_text_from_editor(self):
        """Save text to file"""
        text = self.text_edit.toPlainText()
        if not text:
            QMessageBox.warning(self, "No Text", "There is no text to save.")
            return
        
        # Generate filename
        default_filename = "Text_"
        if self.current_pdf_filename:
            default_filename += os.path.splitext(self.current_pdf_filename)[0] + "_"
        default_filename += f"page_{self.current_page + 1}.txt"
        
        file_path, _ = QFileDialog.getSaveFileName(
            self, "Save Text", default_filename, "Text Files (*.txt);;All Files (*.*)"
        )
        
        if not file_path:
            return
        
        try:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(text)
            
            self.statusBar().showMessage(f"Text saved to {os.path.basename(file_path)}")
            
        except Exception as e:
            QMessageBox.critical(
                self,
                "Error Saving File",
                f"An error occurred:\n{str(e)}"
            )

    # ═══════════════════════════════════════════════════════════════════
    # SPEECH RECOGNITION METHODS
    # ═══════════════════════════════════════════════════════════════════

    def add_speech_recognition_ui(self):
        """Add speech recognition UI"""
        speech_group = QGroupBox("Speech Recognition")
        speech_layout = QVBoxLayout()
        speech_layout.setContentsMargins(10, 5, 10, 5)
        speech_group.setLayout(speech_layout)
        
        # Top controls
        top_controls = QHBoxLayout()
        
        self.speech_button = QPushButton("Start Dictation")
        self.speech_button.clicked.connect(self.start_speech_recognition)
        top_controls.addWidget(self.speech_button)
        
        self.speech_stop_button = QPushButton("Stop")
        self.speech_stop_button.clicked.connect(self.stop_speech_recognition)
        self.speech_stop_button.setEnabled(False)
        top_controls.addWidget(self.speech_stop_button)
        
        self.speech_progress = QProgressBar()
        self.speech_progress.setRange(0, 100)
        self.speech_progress.setValue(0)
        top_controls.addWidget(self.speech_progress, 1)
        
        speech_layout.addLayout(top_controls)
        
        # Settings
        settings_layout = QHBoxLayout()
        
        settings_layout.addWidget(QLabel("Language:"))
        self.language_combo = QComboBox()
        self.language_combo.addItems(["English (US)", "Hindi"])
        self.language_combo.setCurrentIndex(1)
        self.language_combo.currentIndexChanged.connect(self.change_speech_language)
        settings_layout.addWidget(self.language_combo)
        
        settings_layout.addWidget(QLabel("Timeout:"))
        self.timeout_slider = QSlider(Qt.Horizontal)
        self.timeout_slider.setMinimum(5)
        self.timeout_slider.setMaximum(30)
        self.timeout_slider.setValue(10)
        self.timeout_slider.valueChanged.connect(self.update_speech_timeout)
        settings_layout.addWidget(self.timeout_slider)
        self.speech_timeout_label = QLabel("10s")
        settings_layout.addWidget(self.speech_timeout_label)
        
        settings_layout.addWidget(QLabel("Max duration:"))
        self.phrase_slider = QSlider(Qt.Horizontal)
        self.phrase_slider.setMinimum(5)
        self.phrase_slider.setMaximum(60)
        self.phrase_slider.setValue(10)
        self.phrase_slider.valueChanged.connect(self.update_speech_phrase_limit)
        settings_layout.addWidget(self.phrase_slider)
        self.speech_phrase_label = QLabel("10s")
        settings_layout.addWidget(self.speech_phrase_label)
        
        speech_layout.addLayout(settings_layout)
        self.main_layout.addWidget(speech_group)

    def start_speech_recognition(self):
        """Start speech recognition"""
        if self.speech_worker and self.speech_worker.listening:
            self.statusBar().showMessage("Already listening...")
            return
        
        # Update UI
        self.speech_button.setEnabled(False)
        self.speech_stop_button.setEnabled(True)
        self.speech_progress.setRange(0, 0)
        
        # Create worker
        self.speech_worker = SpeechRecognitionWorker(
            language=self.speech_language,
            timeout=self.speech_timeout,
            phrase_time_limit=self.speech_phrase_limit
        )
        
        # Connect signals
        self.speech_worker.result_ready.connect(self.handle_speech_result)
        self.speech_worker.error_occurred.connect(self.handle_speech_error)
        self.speech_worker.status_update.connect(self.update_speech_status)
        
        self.speech_worker.start()

    def stop_speech_recognition(self):
        """Stop speech recognition"""
        if self.speech_worker and self.speech_worker.listening:
            self.speech_worker.stop()
            self.speech_worker.terminate()
            self.speech_worker.wait(1000)
        
        self.speech_button.setEnabled(True)
        self.speech_stop_button.setEnabled(False)
        self.speech_progress.setRange(0, 100)
        self.speech_progress.setValue(0)
        self.statusBar().showMessage("Speech recognition stopped")

    def handle_speech_result(self, text):
        """Handle recognized speech"""
        self.speech_button.setEnabled(True)
        self.speech_stop_button.setEnabled(False)
        self.speech_progress.setRange(0, 100)
        self.speech_progress.setValue(100)
        
        if text:
            cursor = self.text_edit.textCursor()
            cursor.insertText(text + " ")
            self.text_edit.setTextCursor(cursor)
            self.text_edit.setFocus()
            
            self.statusBar().showMessage(f"Recognized: {text}")
        else:
            self.statusBar().showMessage("No speech recognized")

    def handle_speech_error(self, error_message):
        """Handle speech errors"""
        self.speech_button.setEnabled(True)
        self.speech_stop_button.setEnabled(False)
        self.speech_progress.setRange(0, 100)
        self.speech_progress.setValue(0)
        
        self.statusBar().showMessage(error_message)

    def update_speech_status(self, status_message):
        """Update speech status"""
        self.statusBar().showMessage(status_message)

    def change_speech_language(self, index):
        """Change speech language"""
        languages = {0: "en-US", 1: "hi-IN"}
        self.speech_language = languages.get(index, "en-US")
        self.statusBar().showMessage(f"Speech language: {self.speech_language}")

    def update_speech_timeout(self, value):
        """Update speech timeout"""
        self.speech_timeout = value
        self.speech_timeout_label.setText(f"{value}s")

    def update_speech_phrase_limit(self, value):
        """Update phrase time limit"""
        self.speech_phrase_limit = value
        self.speech_phrase_label.setText(f"{value}s")

    # ═══════════════════════════════════════════════════════════════════
    # FOUNDIR CONTROLS
    # ═══════════════════════════════════════════════════════════════════

    def toggle_foundir(self, enabled):
        """Toggle FoundIR denoising"""
        if enabled and not self.foundir_client.enabled:
            QMessageBox.warning(
                self,
                "FoundIR Unavailable",
                "FoundIR server is not available. Please start the server and reconnect."
            )
            self.foundir_checkbox.setChecked(False)
            return
        
        self.foundir_enabled = enabled
        status = "enabled" if enabled else "disabled"
        self.statusBar().showMessage(f"FoundIR denoising {status}")

    def reconnect_foundir(self):
        """Reconnect to FoundIR server"""
        self.statusBar().showMessage("Reconnecting to FoundIR server...")
        
        if self.foundir_client.check_server_availability():
            self.foundir_checkbox.setEnabled(True)
            self.foundir_reconnect_button.setEnabled(False)
            self.foundir_status_label.setText("✓ Server ready")
            self.foundir_status_label.setStyleSheet("color: green;")
            self.statusBar().showMessage("Connected to FoundIR server")
        else:
            self.statusBar().showMessage("Could not connect to FoundIR server")
            QMessageBox.warning(
                self,
                "Connection Failed",
                "Could not connect to FoundIR server.\n"
                "Make sure the server is running at http://172.18.42.4:5000"
            )

    def closeEvent(self, event):
        """Handle application close"""
        try:
            # Save data
            if self.image_lookup_data:
                self.save_image_lookup_data()
            
            # Clean up workers
            for worker in self.ocr_workers:
                if worker.isRunning():
                    worker.terminate()
                    worker.wait(1000)
            
            # Clean up speech
            if self.speech_worker and self.speech_worker.isRunning():
                self.speech_worker.stop()
                self.speech_worker.terminate()
                self.speech_worker.wait(1000)
            
            # Close PDF
            if self.pdf_document:
                self.pdf_document.close()
            
            # Close FoundIR session
            self.foundir_client.close()
            event.accept()
            
        except Exception as e:
            print(f"Error during cleanup: {e}")
            event.accept()

if __name__ == "__main__":
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    
    viewer = PDFOCRViewer()
    viewer.show()
    sys.exit(app.exec_())